Small Instanz

In [ ]:
# 01_build_graph.py
import osmium, math, gzip, pickle as pkl, time
from collections import defaultdict

KEEP_HIGHWAY = {
    "motorway","motorway_link",
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link", "tertiary","tertiary_link",
}
HIGHWAY_MAP = {
    "motorway":1,"motorway_link":2,"trunk":3,"trunk_link":4,
    "primary":5,"primary_link":6,"secondary":7,"secondary_link":8,
    "tertiary":9,"tertiary_link":10
}

GLOBAL_HIGHWAYS = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link"
}

LOCAL_HIGHWAYS = {
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
}

# Kunden/Depot:
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.413933, 12.291324),
    ("C2",    51.487547, 12.078983),
    ("C3",    51.141971, 11.824094),
    ("C4",    51.208217, 11.960303),
    ("C5",    50.884000, 11.597000),
    ("C6",    50.912463, 12.061472),
    ("C7",    51.811731, 12.228786),
    ("C8",    51.009103, 12.462403),
    ("C9",    51.511895, 11.587541),
    ("C10",   51.055287, 12.110381),
]
PAD = 0.20  

LATS = [p[1] for p in TEST_POINTS]
LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

def in_bbox(lat, lon): 
    return LAT_MIN<=lat<=LAT_MAX and LON_MIN<=lon<=LON_MAX

def near_test_point(lat, lon, radius_m=15000):

    for _, tlat, tlon in TEST_POINTS:

        if haversine(
            lat, lon,
            tlat, tlon
        ) <= radius_m:
            return True

    return False
def haversine(lat1, lon1, lat2, lon2):
    R=6371000.0
    p1,p2 = math.radians(lat1), math.radians(lat2)
    dphi=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

class NodeCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_count=defaultdict(int)
        self.wc=0
        self.t0=time.time()
    def way(self, w):
        self.wc+=1
        if self.wc % 500_000 == 0:
            dt=time.time()-self.t0
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 1 … {self.wc:,} Ways  |  {spd:,.0f} Ways/s  |  t={dt:,.1f}s")
        hwy=w.tags.get("highway")
        if hwy not in KEEP_HIGHWAY: 
            return
        try:
            nodes = list(w.nodes)

            if len(nodes) < 2:
                return

            keep = False

            for n in nodes:

                if not n.location.valid():
                    continue

                lat, lon = n.location.lat, n.location.lon

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            if not keep:
                return

            for n in nodes:
                self.node_count[n.ref] += 1

        except:
            return

class Builder(osmium.SimpleHandler):
    def __init__(self,junc):
        super().__init__()
        self.j=junc; self.nc={}; self.edges=[]
        self.aid=0; self.wc=0; self.t0=time.time(); self.last_edges=0
    def way(self, w):
        self.wc+=1
        if self.wc % 200_000 == 0:
            dt=time.time()-self.t0
            de=self.edges.__len__()-self.last_edges
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 2 … {self.wc:,} Ways | Edges: {len(self.edges):,} | ΔEdges:{de:,} | {spd:,.0f} Ways/s | t={dt:,.1f}s")
            self.last_edges=len(self.edges)
        hwy=w.tags.get("highway")
        tunnel = w.tags.get("tunnel", "no")
        if hwy not in KEEP_HIGHWAY: 
            return
        nodes=list(w.nodes)
        if len(nodes)<2: return
        coords={}; keep=False
        for n in nodes:

            try:

                lat, lon = n.location.lat, n.location.lon

                coords[n.ref] = (lat, lon)

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            except:
                return
        if not keep: return
        seg_start=nodes[0].ref; seg_dist=0.0
        for i in range(len(nodes)-1):
            n1,n2=nodes[i].ref,nodes[i+1].ref
            lat1,lon1=coords[n1]; lat2,lon2=coords[n2]
            seg_dist += haversine(lat1,lon1,lat2,lon2)
            is_jun = (n2 in self.j) or (i==len(nodes)-2)
            if is_jun:
                u,v=seg_start,n2
                self.nc[u]=coords[u]; self.nc[v]=coords[v]
                hcode=HIGHWAY_MAP.get(hwy,12)
                self.edges.append({
                    "arc_id": self.aid,
                    "u": u,
                    "v": v,
                    "dist": seg_dist,
                    "highway": hcode,
                    "tunnel": tunnel
                })
                self.aid += 1
                if w.tags.get("oneway","no") not in ("yes","1","true"):
                    self.edges.append({
                        "arc_id": self.aid,
                        "u": v,
                        "v": u,
                        "dist": seg_dist,
                        "highway": hcode,
                        "tunnel": tunnel
                    })
                    self.aid += 1
                seg_start=n2; seg_dist=0.0

def build(pbf_path, out_path="graph_core.pkl.gz"):
    print("=== BUILD GRAPH ===")
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}]  lon[{LON_MIN:.3f},{LON_MAX:.3f}]  | PAD={PAD}")
    t0=time.time()
    print("Pass 1 … counting junctions")
    c=NodeCounter(); c.apply_file(pbf_path, locations=True)
    t1=time.time()
    junc={n for n,k in c.node_count.items() if k!=2}
    print(f"Pass 1 done: {t1-t0:,.1f}s  | Junctions: {len(junc):,} (unique nodes: {len(c.node_count):,})")
    print("Pass 2 … building edges")
    b=Builder(junc); b.apply_file(pbf_path, locations=True)
    t2=time.time()
    core={"node_coords":b.nc,"arcs":b.edges}
    print(f"Pass 2 done: {t2-t1:,.1f}s  | Nodes: {len(b.nc):,} | Edges: {len(b.edges):,}")
    print(
    f"Edges/Node: "
    f"{len(b.edges)/max(len(b.nc),1):.2f}"
)
    with gzip.open(out_path,"wb") as f: pkl.dump(core,f,protocol=pkl.HIGHEST_PROTOCOL)
    print(f"Saved: {out_path}")
    print(f"Total time: {time.time()-t0:,.1f}s")

if __name__=="__main__":
    build(r"C:\Users\9631\Downloads\germany-latest.osm(1).pbf", "graph_core_small.pkl.gz")


=== BUILD GRAPH ===
BBox lat[50.684,52.012]  lon[11.388,12.662]  | PAD=0.2
Pass 1 … counting junctions
Pass 1 … 500,000 Ways  |  16,373 Ways/s  |  t=30.5s
Pass 1 … 1,000,000 Ways  |  25,589 Ways/s  |  t=39.1s
Pass 1 … 1,500,000 Ways  |  32,067 Ways/s  |  t=46.8s
Pass 1 … 2,000,000 Ways  |  36,795 Ways/s  |  t=54.4s
Pass 1 … 2,500,000 Ways  |  41,037 Ways/s  |  t=60.9s
Pass 1 … 3,000,000 Ways  |  44,561 Ways/s  |  t=67.3s
Pass 1 … 3,500,000 Ways  |  47,657 Ways/s  |  t=73.4s
Pass 1 … 4,000,000 Ways  |  50,360 Ways/s  |  t=79.4s
Pass 1 … 4,500,000 Ways  |  53,757 Ways/s  |  t=83.7s
Pass 1 … 5,000,000 Ways  |  56,864 Ways/s  |  t=87.9s
Pass 1 … 5,500,000 Ways  |  59,611 Ways/s  |  t=92.3s
Pass 1 … 6,000,000 Ways  |  62,263 Ways/s  |  t=96.4s
Pass 1 … 6,500,000 Ways  |  65,607 Ways/s  |  t=99.1s
Pass 1 … 7,000,000 Ways  |  68,721 Ways/s  |  t=101.9s
Pass 1 … 7,500,000 Ways  |  71,614 Ways/s  |  t=104.7s
Pass 1 … 8,000,000 Ways  |  74,239 Ways/s  |  t=107.8s
Pass 1 … 8,500,000 Ways  |  76,7

In [ ]:
# 02_map_accidents.py  —  gz-Pickle Variante (kein pyarrow nötig)
import gzip, pickle as pkl, time
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree

# ===== A) Punkte aus 01 (für BBox) =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.413933, 12.291324),
    ("C2",    51.487547, 12.078983),
    ("C3",    51.141971, 11.824094),
    ("C4",    51.208217, 11.960303),
    ("C5",    50.884000, 11.597000),
    ("C6",    50.912463, 12.061472),
    ("C7",    51.811731, 12.228786),
    ("C8",    51.009103, 12.462403),
    ("C9",    51.511895, 11.587541),
    ("C10",   51.055287, 12.110381),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

BUFFER_M = 30  

# ===== B) I/O-Pfade (anpassen) =====
CORE_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_core_small.pkl.gz"
ACCIDENTS_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Accidents.csv"
OUT_EDGES_PKL_GZ = "accidents_core_small.pkl.gz"
OUT_NODECOORDS_PKL_GZ = "node_coords_small.pkl.gz"

def load_core(path):
    with gzip.open(path,"rb") as f:
        return pkl.load(f)

def load_acc(csv_path):
    t0=time.time()
    df=pd.read_csv(csv_path,sep=";",low_memory=False)
    df.columns=df.columns.str.strip()
    for c in ["XGCSWGS84","YGCSWGS84"]:
        df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","."), errors="coerce")
    df=df.dropna(subset=["XGCSWGS84","YGCSWGS84"])
    df["weight"]=df["UKATEGORIE"].map({1:5,2:3,3:1}).fillna(1)
    g=gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["XGCSWGS84"], df["YGCSWGS84"]), crs="EPSG:4326")
    g=g.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX].copy()
    print(f"Accidents: total after BBox = {len(g):,}  | load+filter {time.time()-t0:.1f}s")
    return g

def build_roads_gdf(core):
    from shapely.geometry import LineString
    rows=[]
    nc=core["node_coords"]; arcs=core["arcs"]
    t0=time.time()
    for i,e in enumerate(arcs,1):
        c1=nc.get(e["u"]); c2=nc.get(e["v"])
        if not c1 or not c2: 
            continue
        rows.append({"arc_id":e["arc_id"],"u":e["u"],"v":e["v"],"dist":e["dist"],"highway":e["highway"], "tunnel": e.get("tunnel", "no"),
                     "geometry":LineString([(c1[1],c1[0]),(c2[1],c2[0])])})
        if i % 1_000_000 == 0:
            print(f"build_roads_gdf: {i:,}/{len(arcs):,} edges assembled")
    gdf=gpd.GeoDataFrame(rows, crs="EPSG:4326")
    print(f"Roads GDF: {len(gdf):,} | build {time.time()-t0:.1f}s")
    return gdf

def map_acc(roads, acc, buf_m=BUFFER_M):
    t0=time.time()
    roads_m=roads.to_crs(32632).copy()
    roads_m["length_m"]=roads_m.geometry.length
    geoms=list(roads_m.geometry.values)
    tree=STRtree(geoms)

    acc_m = acc.to_crs(32632)
    assigns=[]
    for i,(pt,w) in enumerate(zip(acc_m.geometry.values, acc_m["weight"].values),1):
        cands=tree.query(pt.buffer(buf_m))
        if len(cands) > 0:
            best=None; bd=1e9
            for ix in cands:
                ix_int = int(ix)
                d=pt.distance(geoms[ix_int])
                if d<bd:
                    bd=d; best=ix_int
            if (best is not None) and (bd<=buf_m):
                assigns.append((int(roads_m.iloc[best]["arc_id"]), float(w)))
        if i % 200_000 == 0:
            print(f"map_acc: processed {i:,} acc pts  | matches so far {len(assigns):,}")
    print(f"map_acc finished loop: {time.time()-t0:.1f}s | matches {len(assigns):,}")

    if not assigns:
        agg=pd.DataFrame(columns=["arc_id","accidents","weighted_accidents"])
    else:
        df=pd.DataFrame(assigns, columns=["arc_id","weight"])
        agg=df.groupby("arc_id").agg(accidents=("arc_id","count"),
                                     weighted_accidents=("weight","sum")).reset_index()

    out=roads_m[["arc_id","u","v","dist","highway", "tunnel","length_m"]].merge(agg, on="arc_id", how="left")
    out[["accidents","weighted_accidents"]]=out[["accidents","weighted_accidents"]].fillna(0)
    out["accident_score"]=out["weighted_accidents"]/(out["length_m"]+1)
    m=out["accident_score"].max()
    out["accident_score_norm"]= out["accident_score"]/m if m>0 else 0.0

    # kompakte Typen
    out = out.astype({
    "arc_id": "int64",
    "u": "int64",
    "v": "int64",
    "highway": "int16",
    "tunnel": "string",
    "dist": "float32",
    "length_m": "float32",

    "accidents": "float32",
    "weighted_accidents": "float32",

    "accident_score": "float32",
    "accident_score_norm": "float32",
})
    
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    return out

if __name__=="__main__":
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={BUFFER_M}m")
    t_all=time.time()
    t0=time.time(); core=load_core(CORE_PKL_GZ); print(f"load_core {time.time()-t0:.1f}s")
    t0=time.time(); roads=build_roads_gdf(core); print(f"build_roads_gdf {time.time()-t0:.1f}s")
    print("\nRoads GDF Speicherbedarf:")
    print(
        f"{roads.memory_usage(deep=True).sum()/1024**3:.2f} GB"
    )
    t0=time.time(); acc=load_acc(ACCIDENTS_CSV); print(f"load_acc {time.time()-t0:.1f}s")
    t0=time.time(); mapped=map_acc(roads,acc,buf_m=BUFFER_M); print(f"map_acc {time.time()-t0:.1f}s")

    # Speichern als gz-Pickle (robust, keine pyarrow-Abhängigkeit)
    with gzip.open(OUT_EDGES_PKL_GZ,"wb") as f:
        pkl.dump(mapped.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
    with gzip.open(OUT_NODECOORDS_PKL_GZ,"wb") as f:
        pkl.dump(core["node_coords"], f, protocol=pkl.HIGHEST_PROTOCOL)

    print(f"Saved: {OUT_EDGES_PKL_GZ}, {OUT_NODECOORDS_PKL_GZ}")
    print(f"Total {time.time()-t_all:.1f}s")


BBox lat[50.684,52.012] lon[11.388,12.662] | PAD=0.2 | buffer=30m
load_core 0.5s
Roads GDF: 412,006 | build 4.6s
build_roads_gdf 4.6s

Roads GDF Speicherbedarf:
0.02 GB
Accidents: total after BBox = 8,482  | load+filter 2.0s
load_acc 2.0s
map_acc finished loop: 2.3s | matches 5,374
u min: 486186
u max: 13867816969
v min: 486186
v max: 13867816969
map_acc 2.5s
Saved: accidents_core_small.pkl.gz, node_coords_small.pkl.gz
Total 13.7s


In [3]:
# 03_map_population_and_merge.py
import time, gzip, pickle as pkl
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import numpy as np

# ===== A) Punkte/BBox konsistent zu 01/02 =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.413933, 12.291324),
    ("C2",    51.487547, 12.078983),
    ("C3",    51.141971, 11.824094),
    ("C4",    51.208217, 11.960303),
    ("C5",    50.884000, 11.597000),
    ("C6",    50.912463, 12.061472),
    ("C7",    51.811731, 12.228786),
    ("C8",    51.009103, 12.462403),
    ("C9",    51.511895, 11.587541),
    ("C10",   51.055287, 12.110381),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

# ===== B) I/O-Pfade =====
ACCIDENTS_GZ_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\accidents_core_small.pkl.gz"  # Output aus 02 (gz-Pickle)
NODE_COORDS_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_small.pkl.gz"   # aus 02
POP_GPKG = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\ESTAT_Census_2021_V2.gpkg"
CHARGING_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Ladesaeulenregister_BNetzA_2026-04-22.csv"
OUT_PARQUET = "graph_final_core_small.parquet"  # Parquet mit Fallback
OUT_GZ_PKL = "graph_final_core_small.pkl.gz"    # Fallback-Ziel
OUT_CHARGING_NODES = "charging_nodes_small.parquet"
OUT_CHARGING_HUBS = "charging_hubs.small.parquet"

gdf_pop_all = gpd.read_file(POP_GPKG)

print("CRS:", gdf_pop_all.crs)
print("Anzahl Zeilen:", len(gdf_pop_all))
print("Bounds:", gdf_pop_all.total_bounds)
print(gdf_pop_all.columns)

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def map_charging_to_nodes(node_coords, charging_csv_path, min_power_kw=150, max_snap_m=150):
    node_ids = np.array(list(node_coords.keys()))
    node_lats = np.array([node_coords[n][0] for n in node_ids])
    node_lons = np.array([node_coords[n][1] for n in node_ids])

    charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)
    charging_df.columns = charging_df.columns.str.strip()

    charging_df = charging_df[[
        "Ladeeinrichtungs-ID",
        "Breitengrad",
        "Längengrad",
        "Nennleistung Ladeeinrichtung [kW]"
    ]].dropna().copy()

    charging_df.columns = ["id", "lat", "lon", "power_kw"]

    charging_df["lat"] = charging_df["lat"].astype(str).str.replace(",", ".").astype(float)
    charging_df["lon"] = charging_df["lon"].astype(str).str.replace(",", ".").astype(float)
    charging_df["power_kw"] = charging_df["power_kw"].astype(str).str.replace(",", ".").astype(float)

    charging_df = charging_df[charging_df["power_kw"] >= min_power_kw].copy()

    print(f"Schnelllader >= {min_power_kw} kW: {len(charging_df):,}")

    charging_nodes = []

    for _, row in charging_df.iterrows():
        dists = haversine_vectorized(row["lat"], row["lon"], node_lats, node_lons)
        idx = np.argmin(dists)

        node_id = int(node_ids[idx])
        snap_dist = float(dists[idx])

        charging_nodes.append({
            "charger_id": row["id"],
            "node": node_id,
            "lat": row["lat"],
            "lon": row["lon"],
            "power_kw": row["power_kw"],
            "snap_dist_m": snap_dist
        })

    charging_nodes_df = pd.DataFrame(charging_nodes)
    charging_nodes_df = charging_nodes_df[charging_nodes_df["snap_dist_m"] <= max_snap_m].copy()

    print(f"Nach Snap-Filter <= {max_snap_m} m: {len(charging_nodes_df):,}")

    charging_hubs_df = charging_nodes_df.groupby("node").agg(
        max_power_kw=("power_kw", "max"),
        n_chargers=("charger_id", "count"),
        min_snap_dist_m=("snap_dist_m", "min")
    ).reset_index()

    print(f"Einzigartige Ladeknoten: {len(charging_hubs_df):,}")

    return charging_nodes_df, charging_hubs_df

def build_temp_roads_geometry(edges_df, node_coords):
    t0=time.time()
    rows=[]
    for r in edges_df[["arc_id","u","v"]].itertuples(index=False):
        c1=node_coords.get(r.u); c2=node_coords.get(r.v)
        if c1 and c2:
            rows.append((r.arc_id, LineString([(c1[1],c1[0]), (c2[1],c2[0])])))
    gdf=gpd.GeoDataFrame(rows, columns=["arc_id","geometry"], crs="EPSG:4326")
    print(f"Temp roads geometry: {len(gdf):,} | {time.time()-t0:.1f}s")
    return gdf

def map_population(edges_gz_pkl, pop_gpkg, buffer_m=50):
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={buffer_m}m")
    t_all=time.time()

    # 1) Edges aus gz-Pickle laden
    t0=time.time()
    with gzip.open(edges_gz_pkl,"rb") as f:
        edges = pd.DataFrame(pkl.load(f))
    print(f"read edges (gz-pkl) {time.time()-t0:.1f}s  | {len(edges):,} rows")

    # 2) Node-Coords laden
    with gzip.open(NODE_COORDS_PKL_GZ,"rb") as f:
        node_coords=pkl.load(f)

    # 3) temporäre Linien für Overlay
    gdf_roads = build_temp_roads_geometry(edges, node_coords).to_crs(3035)
    print(
    "Roads:",
    len(gdf_roads)
)
    gdf_roads["geom_buf"] = gdf_roads.geometry.buffer(buffer_m)
    gdf_roads = gdf_roads.set_geometry("geom_buf")

    # 4) Bevölkerung lesen + BBox-Filter in EPSG:4326
    t0=time.time()
    gdf_pop_all = gpd.read_file(pop_gpkg)[["T","geometry"]].rename(
    columns={"T":"population"}
    )

    gdf_pop_all = gdf_pop_all[
        gdf_pop_all["population"] > 0
    ]

    # zuerst nach WGS84 transformieren
    gdf_pop_all = gdf_pop_all.to_crs(4326)

    # dann BBox-Filter
    gdf_pop_all = gdf_pop_all.cx[
        LON_MIN:LON_MAX,
        LAT_MIN:LAT_MAX
    ].copy()

    # anschließend zurück nach 3035
    gdf_pop = gdf_pop_all.to_crs(3035)

    print(
    "Pop cells:",
    len(gdf_pop)
)
    gdf_pop["cell_area"]=gdf_pop.geometry.area
    print(f"pop cells after BBox: {len(gdf_pop):,}  | load+filter {time.time()-t0:.1f}s")

    # 5) Overlay
    t0=time.time()
    ov=gpd.overlay(gdf_roads, gdf_pop, how="intersection")
    print(f"overlay rows: {len(ov):,} | {time.time()-t0:.1f}s")

    if len(ov)>0:
        ov["overlap_area"]=ov.geometry.area
        ov["assigned_pop"]= ov["population"]*(ov["overlap_area"]/ov["cell_area"])
        pop_agg = ov.groupby("arc_id").agg(total_pop=("assigned_pop","sum")).reset_index()
    else:
        pop_agg = pd.DataFrame(columns=["arc_id","total_pop"])

    # 6) Merge + Scores
    out = edges.merge(pop_agg, on="arc_id", how="left")

    out["total_pop"] = (
        out["total_pop"]
        .fillna(0)
        .astype("float32")
    )

    out["pop_per_m"] = (
        out["total_pop"] /
        (out["length_m"] + 1)
    ).astype("float32")

    m = out["pop_per_m"].max()

    if m > 0:
        out["pop_score_norm"] = (
            out["pop_per_m"] / m
        ).astype("float32")
    else:
        out["pop_score_norm"] = out["pop_per_m"]

    # ==========================================================
    # 7) Datentypen
    # ==========================================================

    int_cols = {
        "arc_id": "int64",
        "u": "int64",
        "v": "int64",
        "highway": "int16",
    }

    if "tunnel" in out.columns:
        out["tunnel"] = out["tunnel"].astype("category")

    for col, dtype in int_cols.items():
        if col in out.columns:
            out[col] = out[col].astype(dtype)

    float_cols = [
        "dist",
        "length_m",
        "accidents",
        "weighted_accidents",
        "accident_score_norm",
        "total_pop",
        "pop_per_m",
        "pop_score_norm",
    ]

    for col in float_cols:
        if col in out.columns:
            out[col] = out[col].astype("float32")

    # Hilfsspalte entfernen
    if "accident_score" in out.columns:
        out = out.drop(columns=["accident_score"])

    # ==========================================================
    # DEBUG
    # ==========================================================

    print("\n===== ID CHECK =====")
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    print("\n===== DTYPES =====")
    print(out[["arc_id", "u", "v", "highway"]].dtypes)

    print("\n===== POPULATION =====")
    print("total_pop max:", out["total_pop"].max())
    print("pop_per_m max:", out["pop_per_m"].max())
    print("pop_score_norm max:", out["pop_score_norm"].max())

    print(f"\nmap_population: total {time.time()-t_all:.1f}s")

    return out


if __name__=="__main__":
    merged = map_population(ACCIDENTS_GZ_PKL, POP_GPKG, buffer_m=50)
    with gzip.open(NODE_COORDS_PKL_GZ, "rb") as f:
        node_coords = pkl.load(f)

    charging_nodes_df, charging_hubs_df = map_charging_to_nodes(
        node_coords,
        CHARGING_CSV,
        min_power_kw=150,
        max_snap_m=150
    )

    charging_nodes_df.to_parquet(
        OUT_CHARGING_NODES,
        compression="zstd",
        index=False
    )

    charging_hubs_df.to_parquet(
        OUT_CHARGING_HUBS,
        compression="zstd",
        index=False
    )

    print("Saved:", OUT_CHARGING_NODES)
    print("Saved:", OUT_CHARGING_HUBS)

    # Parquet schreiben (schnell/kompakt); bei Engine-Problemen Fallback auf gz-Pickle
    try:
        merged.to_parquet(OUT_PARQUET, compression="zstd", index=False)
        print("Saved:", OUT_PARQUET)
    except Exception as e:
        print("Parquet write failed, fallback to gz-Pickle:", e)
        with gzip.open(OUT_GZ_PKL,"wb") as f:
            pkl.dump(merged.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
        print("Saved:", OUT_GZ_PKL)


CRS: EPSG:3035
Anzahl Zeilen: 4595749
Bounds: [ 943000.  941000. 6505000. 5416000.]
Index(['GRD_ID', 'T', 'M', 'F', 'Y_LT15', 'Y_1564', 'Y_GE65', 'EMP', 'NAT',
       'EU_OTH', 'OTH', 'SAME', 'CHG_IN', 'CHG_OUT', 'LAND_SURFACE',
       'POPULATED', 'T_CI', 'M_CI', 'F_CI', 'Y_LT15_CI', 'Y_1564_CI',
       'Y_GE65_CI', 'EMP_CI', 'NAT_CI', 'EU_OTH_CI', 'OTH_CI', 'SAME_CI',
       'CHG_IN_CI', 'CHG_OUT_CI', 'geometry'],
      dtype='str')
BBox lat[50.684,52.012] lon[11.388,12.662] | PAD=0.2 | buffer=50m
read edges (gz-pkl) 1.4s  | 412,006 rows
Temp roads geometry: 412,006 | 4.0s
Roads: 412006
Pop cells: 7131
pop cells after BBox: 7,131  | load+filter 44.1s
overlay rows: 428,130 | 17.5s

===== ID CHECK =====
u min: 486186
u max: 13867816969
v min: 486186
v max: 13867816969

===== DTYPES =====
arc_id     int64
u          int64
v          int64
highway    int16
dtype: object

===== POPULATION =====
total_pop max: 373.195
pop_per_m max: 85.1737
pop_score_norm max: 1.0

map_population: total 73

C:\Users\9631\AppData\Local\Temp\ipykernel_9628\3739738085.py:63: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)


Schnelllader >= 150 kW: 21,657
Nach Snap-Filter <= 150 m: 514
Einzigartige Ladeknoten: 212
Saved: charging_nodes_small.parquet
Saved: charging_hubs.small.parquet
Saved: graph_final_core_small.parquet


In [ ]:
# 04_dijkstra_matrix.py — robust (Parquet → gz-Pickle Fallback) + sicherer nearest_nodes
import time, math, gzip, pickle as pkl
import pandas as pd
import numpy as np
import networkx as nx
 
# ===== I/O: finales Kantenformat aus 03 (Parquet ODER gz-Pickle) =====
# Wenn du in 03 Parquet geschrieben hast: Pfad auf *.parquet setzen.
# Wenn 03 auf gz-Pickle gefallen ist: Pfad auf *.pkl.gz setzen — der Reader unten kann beides.
GRAPH_FILE = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_small.parquet"  # Parquet oder gz-Pickle
NODE_COORDS_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_small.pkl.gz"
 
POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.413933, "lon": 12.291324, "type": "customer"},
    {"id": "C2",    "lat": 51.487547, "lon": 12.078983, "type": "customer"},
    {"id": "C3",    "lat": 51.141971, "lon": 11.824094, "type": "customer"},
    {"id": "C4",    "lat": 51.208217, "lon": 11.960303, "type": "customer"},
    {"id": "C5",    "lat": 50.884000, "lon": 11.597000, "type": "customer"},
    {"id": "C6",    "lat": 50.912463, "lon": 12.061472, "type": "customer"},
    {"id": "C7",    "lat": 51.811731, "lon": 12.228786, "type": "customer"},
    {"id": "C8",    "lat": 51.009103, "lon": 12.462403, "type": "customer"},
    {"id": "C9",    "lat": 51.511895, "lon": 11.587541, "type": "customer"},
    {"id": "C10",   "lat": 51.055287, "lon": 12.110381, "type": "customer"},
]

 
LOAD_STATES = {
    "loaded": True,
    "empty": False,
}
 
# ===== Profile / Gewichte =====
PROFILES = {
    "shortest": {
        "mode": "distance"
    },
 
    "safest": {
        "mode": "risk"
    },
 
    "fastest": {
        "mode": "time"
    }
}
 
W_ACC = 0.5
W_POP = 0.5
 
SPEED_KMH = {
    "motorway":80,"motorway_link":50,"trunk":70,"trunk_link":45,"primary":60,"primary_link":40,
    "secondary":50,"secondary_link":35,"tertiary":40,"tertiary_link":30,"residential":25,"unclassified":30
}
ROAD_PENALTY = {
    "motorway":0.0,"motorway_link":0.05,"trunk":0.08,"trunk_link":0.12,"primary":0.20,"primary_link":0.25,
    "secondary":0.35,"secondary_link":0.40,"tertiary":0.55,"tertiary_link":0.60,"residential":0.90,"unclassified":0.70
}
HIGHWAY_REV = {
    1:"motorway",2:"motorway_link",3:"trunk",4:"trunk_link",5:"primary",6:"primary_link",7:"secondary",
    8:"secondary_link",9:"tertiary",10:"tertiary_link",11:"residential",12:"unclassified"
}
 
def load_graph():
    t0 = time.time()
    # Versuche Parquet; wenn fehlschlägt (oder Datei ist pkl.gz): Fallback
    try:
        edges = pd.read_parquet(GRAPH_FILE)
        print(f"read graph (parquet): {len(edges):,} rows")
    except Exception as e:
        print("Parquet read failed or not a parquet file, fallback to gz-Pickle:", e)
        with gzip.open(GRAPH_FILE, "rb") as f:
            edges = pd.DataFrame(pkl.load(f))
        print(f"read graph (gz-pkl): {len(edges):,} rows")
 
    with gzip.open(NODE_COORDS_PKL,"rb") as f:
        node_coords = pkl.load(f)
    print(f"node_coords: {len(node_coords):,} | load {time.time()-t0:.1f}s")
 
    G = nx.DiGraph()
    for e in edges.itertuples(index=False):
        u = int(e.u); v = int(e.v)
        dist_km = float(e.dist) / 1000.0
        hwy = HIGHWAY_REV.get(int(e.highway), "unclassified")
 
        sp = SPEED_KMH.get(hwy, 30)
        time_h = dist_km / sp if sp > 0 else 999.0
 
        acc = float(getattr(e, "accident_score_norm", 0.0))
        pop = float(getattr(e, "pop_score_norm", 0.0))
        base = W_ACC*acc + W_POP*pop
        risk = base * dist_km
 
        road = ROAD_PENALTY.get(hwy, 0.7) * dist_km
 
        G.add_edge(u, v, dist_km=dist_km, time_h=time_h,
                   risk_cost=risk, road_penalty_cost=road, highway=hwy, tunnel=getattr(e, "tunnel", "no"))
 
    print(f"nx graph: nodes={G.number_of_nodes():,} edges={G.number_of_edges():,}")
    return G, node_coords
 
def nearest_nodes(points, node_coords, valid):
    # Nur Knoten verwenden, die sowohl in valid als auch in node_coords enthalten sind
    nodes = [n for n in valid if n in node_coords]
    if not nodes:
        raise RuntimeError("Keine gültigen Knoten mit Koordinaten nach Komponentenreduktion gefunden.")
    coords = np.array([node_coords[n] for n in nodes], dtype=float)  # (N, 2) = (lat, lon)
 
    out = []
    for p in points:
        lat, lon = float(p["lat"]), float(p["lon"])
        # euklidischer Proxy in (lat,lon) — für DE ok
        d = np.sum((coords - np.array([lat, lon]))**2, axis=1)
        idx = int(np.argmin(d))
        n = nodes[idx]
        out.append({**p, "node": n})
    return pd.DataFrame(out)
 
def make_weight(profile, loaded=True):
 
    mode = PROFILES[profile]["mode"]
 
    def w(u, v, d):
 
        # Tunnelverbot für beladenes Fahrzeug
        if loaded and d.get("tunnel", "no") != "no":
            return float("inf")
 
        if mode == "distance":
            return d["dist_km"]
 
        elif mode == "time":
            return d["time_h"]
 
        elif mode == "risk":
            return d["risk_cost"]
 
        return d["dist_km"]
 
    return w
 
def compute(G, s, t, prof, loaded=True):
    try:
        cost, path = nx.single_source_dijkstra(
            G, s, t,
            weight=make_weight(prof, loaded=loaded)
        )
 
        real_dist_km = sum(
            G[path[i]][path[i+1]]["dist_km"]
            for i in range(len(path)-1)
        )
 
        time_min = sum(
            G[path[i]][path[i+1]]["time_h"]
            for i in range(len(path)-1)
        ) * 60.0
 
        
        risk_total = sum(
            G[path[i]][path[i+1]]["risk_cost"]
            for i in range(len(path)-1)
        )
 
        road_penalty_total = sum(
            G[path[i]][path[i+1]]["road_penalty_cost"]
            for i in range(len(path)-1)
        )
 
        tunnel_used = False
 
        for i in range(len(path)-1):
 
            if G[path[i]][path[i+1]].get("tunnel", "no") != "no":
                tunnel_used = True
 
        return (
            True,
            real_dist_km,
            time_min,
            risk_total,
            road_penalty_total,
            cost,
            path,
            tunnel_used
        )
 
    except Exception:
        return (
            False,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            [],
            False
        )
 
if __name__ == "__main__":
    t_all = time.time()
 
    G, node_coords = load_graph()
 
    # ggf. auf größte Komponente reduzieren
    if not nx.is_weakly_connected(G):
        largest = max(nx.weakly_connected_components(G), key=len)
        G = G.subgraph(largest).copy()
        print(f"weakly connected -> reduced to nodes={len(G.nodes()):,}, edges={len(G.edges()):,}")
 
    valid = set(G.nodes())
 
    # optional: node_coords direkt auf gültige Knoten beschränken (schneller/kleiner)
    node_coords = {k: v for k, v in node_coords.items() if k in valid}
 
    pts_df = nearest_nodes(POINTS, node_coords, valid)
    print("mapped points:", [(r["id"], r["node"]) for _, r in pts_df.iterrows()])
 
    rows = []
    t0 = time.time()
    for s in pts_df.itertuples():
 
        for t in pts_df.itertuples():
 
            if s.id == t.id:
                continue
 
            for prof in PROFILES:
 
                for load_name, loaded in LOAD_STATES.items():
 
                    ok, dkm, tmin, risk_total, road_penalty_total, cost, path, tunnel_used = compute(
                        G,
                        s.node,
                        t.node,
                        prof,
                        loaded=loaded
                    )
 
                    rows.append({
                        "from": s.id,
                        "to": t.id,
 
                        "profile": prof,
                        "load_state": load_name,
 
                        "dist_km": dkm,
                        "cost": cost,
                        "time_min": tmin,
                        "risk_total": risk_total,
                        "road_penalty_total": road_penalty_total,
 
                        "reachable": ok,
 
                        "path_len": len(path),
                        "path": path,
                        "tunnel_used": tunnel_used
                    })
    print(f"OD computed in {time.time()-t0:.1f}s  | rows={len(rows):,}")
 
    pd.DataFrame(rows).to_csv("od_matrix_small.csv", index=False)
    pts_df.to_csv("mapped_points_clean.csv", index=False)
    with gzip.open("routes_clean.pkl.gz","wb") as f:
        pkl.dump(rows, f, protocol=pkl.HIGHEST_PROTOCOL)
 
    print("Saved od_matrix_small.csv  | total", f"{time.time()-t_all:.1f}s")

read graph (parquet): 412,006 rows
node_coords: 232,865 | load 0.3s
nx graph: nodes=232,865 edges=412,006
weakly connected -> reduced to nodes=222,261, edges=392,629
mapped points: [('DEPOT', 3705572847), ('C1', 2597074139), ('C2', 7873168147), ('C3', 6957502365), ('C4', 13476349607), ('C5', 617520464), ('C6', 264785942), ('C7', 1376131970), ('C8', 7972583316), ('C9', 4582536101), ('C10', 13378931612)]
OD computed in 205.5s  | rows=660
Saved od_matrix_small.csv  | total 211.7s


Medium Instanz

In [ ]:
# 01_build_graph.py
import osmium, math, gzip, pickle as pkl, time
from collections import defaultdict

KEEP_HIGHWAY = {
    "motorway","motorway_link",
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link", "tertiary","tertiary_link",
}
HIGHWAY_MAP = {
    "motorway":1,"motorway_link":2,"trunk":3,"trunk_link":4,
    "primary":5,"primary_link":6,"secondary":7,"secondary_link":8,
    "tertiary":9,"tertiary_link":10
}

GLOBAL_HIGHWAYS = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link"
}

LOCAL_HIGHWAYS = {
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
}

# Kunden/Depot:
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
]
PAD = 0.20 

LATS = [p[1] for p in TEST_POINTS]
LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

def in_bbox(lat, lon): 
    return LAT_MIN<=lat<=LAT_MAX and LON_MIN<=lon<=LON_MAX

def near_test_point(lat, lon, radius_m=15000):

    for _, tlat, tlon in TEST_POINTS:

        if haversine(
            lat, lon,
            tlat, tlon
        ) <= radius_m:
            return True

    return False
def haversine(lat1, lon1, lat2, lon2):
    R=6371000.0
    p1,p2 = math.radians(lat1), math.radians(lat2)
    dphi=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

class NodeCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_count=defaultdict(int)
        self.wc=0
        self.t0=time.time()
    def way(self, w):
        self.wc+=1
        if self.wc % 500_000 == 0:
            dt=time.time()-self.t0
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 1 … {self.wc:,} Ways  |  {spd:,.0f} Ways/s  |  t={dt:,.1f}s")
        hwy=w.tags.get("highway")
        if hwy not in KEEP_HIGHWAY: 
            return
        try:
            nodes = list(w.nodes)

            if len(nodes) < 2:
                return

            keep = False

            for n in nodes:

                if not n.location.valid():
                    continue

                lat, lon = n.location.lat, n.location.lon

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            if not keep:
                return

            for n in nodes:
                self.node_count[n.ref] += 1

        except:
            return

class Builder(osmium.SimpleHandler):
    def __init__(self,junc):
        super().__init__()
        self.j=junc; self.nc={}; self.edges=[]
        self.aid=0; self.wc=0; self.t0=time.time(); self.last_edges=0
    def way(self, w):
        self.wc+=1
        if self.wc % 200_000 == 0:
            dt=time.time()-self.t0
            de=self.edges.__len__()-self.last_edges
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 2 … {self.wc:,} Ways | Edges: {len(self.edges):,} | ΔEdges:{de:,} | {spd:,.0f} Ways/s | t={dt:,.1f}s")
            self.last_edges=len(self.edges)
        hwy=w.tags.get("highway")
        tunnel = w.tags.get("tunnel", "no")
        if hwy not in KEEP_HIGHWAY: 
            return
        nodes=list(w.nodes)
        if len(nodes)<2: return
        coords={}; keep=False
        for n in nodes:

            try:

                lat, lon = n.location.lat, n.location.lon

                coords[n.ref] = (lat, lon)

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            except:
                return
        if not keep: return
        seg_start=nodes[0].ref; seg_dist=0.0
        for i in range(len(nodes)-1):
            n1,n2=nodes[i].ref,nodes[i+1].ref
            lat1,lon1=coords[n1]; lat2,lon2=coords[n2]
            seg_dist += haversine(lat1,lon1,lat2,lon2)
            is_jun = (n2 in self.j) or (i==len(nodes)-2)
            if is_jun:
                u,v=seg_start,n2
                self.nc[u]=coords[u]; self.nc[v]=coords[v]
                hcode=HIGHWAY_MAP.get(hwy,12)
                self.edges.append({
                    "arc_id": self.aid,
                    "u": u,
                    "v": v,
                    "dist": seg_dist,
                    "highway": hcode,
                    "tunnel": tunnel
                })
                self.aid += 1
                if w.tags.get("oneway","no") not in ("yes","1","true"):
                    self.edges.append({
                        "arc_id": self.aid,
                        "u": v,
                        "v": u,
                        "dist": seg_dist,
                        "highway": hcode,
                        "tunnel": tunnel
                    })
                    self.aid += 1
                seg_start=n2; seg_dist=0.0

def build(pbf_path, out_path="graph_core.pkl.gz"):
    print("=== BUILD GRAPH ===")
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}]  lon[{LON_MIN:.3f},{LON_MAX:.3f}]  | PAD={PAD}")
    t0=time.time()
    print("Pass 1 … counting junctions")
    c=NodeCounter(); c.apply_file(pbf_path, locations=True)
    t1=time.time()
    junc={n for n,k in c.node_count.items() if k!=2}
    print(f"Pass 1 done: {t1-t0:,.1f}s  | Junctions: {len(junc):,} (unique nodes: {len(c.node_count):,})")
    print("Pass 2 … building edges")
    b=Builder(junc); b.apply_file(pbf_path, locations=True)
    t2=time.time()
    core={"node_coords":b.nc,"arcs":b.edges}
    print(f"Pass 2 done: {t2-t1:,.1f}s  | Nodes: {len(b.nc):,} | Edges: {len(b.edges):,}")
    print(
    f"Edges/Node: "
    f"{len(b.edges)/max(len(b.nc),1):.2f}"
)
    with gzip.open(out_path,"wb") as f: pkl.dump(core,f,protocol=pkl.HIGHEST_PROTOCOL)
    print(f"Saved: {out_path}")
    print(f"Total time: {time.time()-t0:,.1f}s")

if __name__=="__main__":
    build(r"C:\Users\9631\Downloads\germany-latest.osm(1).pbf", "graph_core_medium.pkl.gz")


=== BUILD GRAPH ===
BBox lat[50.105,52.302]  lon[10.592,14.564]  | PAD=0.2
Pass 1 … counting junctions
Pass 1 … 500,000 Ways  |  14,585 Ways/s  |  t=34.3s
Pass 1 … 1,000,000 Ways  |  21,452 Ways/s  |  t=46.6s
Pass 1 … 1,500,000 Ways  |  26,455 Ways/s  |  t=56.7s
Pass 1 … 2,000,000 Ways  |  30,050 Ways/s  |  t=66.6s
Pass 1 … 2,500,000 Ways  |  33,519 Ways/s  |  t=74.6s
Pass 1 … 3,000,000 Ways  |  36,257 Ways/s  |  t=82.7s
Pass 1 … 3,500,000 Ways  |  38,511 Ways/s  |  t=90.9s
Pass 1 … 4,000,000 Ways  |  41,141 Ways/s  |  t=97.2s
Pass 1 … 4,500,000 Ways  |  43,594 Ways/s  |  t=103.2s
Pass 1 … 5,000,000 Ways  |  45,749 Ways/s  |  t=109.3s
Pass 1 … 5,500,000 Ways  |  47,609 Ways/s  |  t=115.5s
Pass 1 … 6,000,000 Ways  |  49,705 Ways/s  |  t=120.7s
Pass 1 … 6,500,000 Ways  |  52,386 Ways/s  |  t=124.1s
Pass 1 … 7,000,000 Ways  |  54,888 Ways/s  |  t=127.5s
Pass 1 … 7,500,000 Ways  |  56,655 Ways/s  |  t=132.4s
Pass 1 … 8,000,000 Ways  |  58,060 Ways/s  |  t=137.8s
Pass 1 … 8,500,000 Ways  | 

In [ ]:
# 02_map_accidents.py  —  gz-Pickle Variante (kein pyarrow nötig)
import gzip, pickle as pkl, time
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree

# ===== A) Punkte aus 01 (für BBox) =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

BUFFER_M = 30  

# ===== B) I/O-Pfade (anpassen) =====
CORE_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_core_medium.pkl.gz"
ACCIDENTS_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Accidents.csv"
OUT_EDGES_PKL_GZ = "accidents_core_medium.pkl.gz"
OUT_NODECOORDS_PKL_GZ = "node_coords_medium.pkl.gz"

def load_core(path):
    with gzip.open(path,"rb") as f:
        return pkl.load(f)

def load_acc(csv_path):
    t0=time.time()
    df=pd.read_csv(csv_path,sep=";",low_memory=False)
    df.columns=df.columns.str.strip()
    for c in ["XGCSWGS84","YGCSWGS84"]:
        df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","."), errors="coerce")
    df=df.dropna(subset=["XGCSWGS84","YGCSWGS84"])
    df["weight"]=df["UKATEGORIE"].map({1:5,2:3,3:1}).fillna(1)
    g=gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["XGCSWGS84"], df["YGCSWGS84"]), crs="EPSG:4326")
    g=g.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX].copy()
    print(f"Accidents: total after BBox = {len(g):,}  | load+filter {time.time()-t0:.1f}s")
    return g

def build_roads_gdf(core):
    from shapely.geometry import LineString
    rows=[]
    nc=core["node_coords"]; arcs=core["arcs"]
    t0=time.time()
    for i,e in enumerate(arcs,1):
        c1=nc.get(e["u"]); c2=nc.get(e["v"])
        if not c1 or not c2: 
            continue
        rows.append({"arc_id":e["arc_id"],"u":e["u"],"v":e["v"],"dist":e["dist"],"highway":e["highway"], "tunnel": e.get("tunnel", "no"),
                     "geometry":LineString([(c1[1],c1[0]),(c2[1],c2[0])])})
        if i % 1_000_000 == 0:
            print(f"build_roads_gdf: {i:,}/{len(arcs):,} edges assembled")
    gdf=gpd.GeoDataFrame(rows, crs="EPSG:4326")
    print(f"Roads GDF: {len(gdf):,} | build {time.time()-t0:.1f}s")
    return gdf

def map_acc(roads, acc, buf_m=BUFFER_M):
    t0=time.time()
    roads_m=roads.to_crs(32632).copy()
    roads_m["length_m"]=roads_m.geometry.length
    geoms=list(roads_m.geometry.values)
    tree=STRtree(geoms)

    acc_m = acc.to_crs(32632)
    assigns=[]
    for i,(pt,w) in enumerate(zip(acc_m.geometry.values, acc_m["weight"].values),1):
        cands=tree.query(pt.buffer(buf_m))
        if len(cands) > 0:
            best=None; bd=1e9
            for ix in cands:
                ix_int = int(ix)
                d=pt.distance(geoms[ix_int])
                if d<bd:
                    bd=d; best=ix_int
            if (best is not None) and (bd<=buf_m):
                assigns.append((int(roads_m.iloc[best]["arc_id"]), float(w)))
        if i % 200_000 == 0:
            print(f"map_acc: processed {i:,} acc pts  | matches so far {len(assigns):,}")
    print(f"map_acc finished loop: {time.time()-t0:.1f}s | matches {len(assigns):,}")

    if not assigns:
        agg=pd.DataFrame(columns=["arc_id","accidents","weighted_accidents"])
    else:
        df=pd.DataFrame(assigns, columns=["arc_id","weight"])
        agg=df.groupby("arc_id").agg(accidents=("arc_id","count"),
                                     weighted_accidents=("weight","sum")).reset_index()

    out=roads_m[["arc_id","u","v","dist","highway", "tunnel","length_m"]].merge(agg, on="arc_id", how="left")
    out[["accidents","weighted_accidents"]]=out[["accidents","weighted_accidents"]].fillna(0)
    out["accident_score"]=out["weighted_accidents"]/(out["length_m"]+1)
    m=out["accident_score"].max()
    out["accident_score_norm"]= out["accident_score"]/m if m>0 else 0.0

    # kompakte Typen
    out = out.astype({
    "arc_id": "int64",
    "u": "int64",
    "v": "int64",
    "highway": "int16",
    "tunnel": "string",
    "dist": "float32",
    "length_m": "float32",

    "accidents": "float32",
    "weighted_accidents": "float32",

    "accident_score": "float32",
    "accident_score_norm": "float32",
})
    
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    return out

if __name__=="__main__":
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={BUFFER_M}m")
    t_all=time.time()
    t0=time.time(); core=load_core(CORE_PKL_GZ); print(f"load_core {time.time()-t0:.1f}s")
    t0=time.time(); roads=build_roads_gdf(core); print(f"build_roads_gdf {time.time()-t0:.1f}s")
    print("\nRoads GDF Speicherbedarf:")
    print(
        f"{roads.memory_usage(deep=True).sum()/1024**3:.2f} GB"
    )
    t0=time.time(); acc=load_acc(ACCIDENTS_CSV); print(f"load_acc {time.time()-t0:.1f}s")
    t0=time.time(); mapped=map_acc(roads,acc,buf_m=BUFFER_M); print(f"map_acc {time.time()-t0:.1f}s")

    # Speichern als gz-Pickle (robust, keine pyarrow-Abhängigkeit)
    with gzip.open(OUT_EDGES_PKL_GZ,"wb") as f:
        pkl.dump(mapped.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
    with gzip.open(OUT_NODECOORDS_PKL_GZ,"wb") as f:
        pkl.dump(core["node_coords"], f, protocol=pkl.HIGHEST_PROTOCOL)

    print(f"Saved: {OUT_EDGES_PKL_GZ}, {OUT_NODECOORDS_PKL_GZ}")
    print(f"Total {time.time()-t_all:.1f}s")


BBox lat[50.105,52.302] lon[10.592,14.564] | PAD=0.2 | buffer=30m
load_core 1.1s
build_roads_gdf: 1,000,000/1,070,450 edges assembled
Roads GDF: 1,070,450 | build 12.2s
build_roads_gdf 12.4s

Roads GDF Speicherbedarf:
0.06 GB
Accidents: total after BBox = 26,518  | load+filter 2.0s
load_acc 2.0s
map_acc finished loop: 7.0s | matches 15,076
u min: 442759
u max: 13868515003
v min: 442759
v max: 13868515003
map_acc 7.4s
Saved: accidents_core_medium.pkl.gz, node_coords_medium.pkl.gz
Total 33.7s


In [7]:
# 03_map_population_and_merge.py
import time, gzip, pickle as pkl
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import numpy as np

# ===== A) Punkte/BBox konsistent zu 01/02 =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

# ===== B) I/O-Pfade =====
ACCIDENTS_GZ_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\accidents_core_medium.pkl.gz"  # Output aus 02 (gz-Pickle)
NODE_COORDS_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_medium.pkl.gz"   # aus 02
POP_GPKG = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\ESTAT_Census_2021_V2.gpkg"
CHARGING_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Ladesaeulenregister_BNetzA_2026-04-22.csv"
OUT_PARQUET = "graph_final_core_medium.parquet"  # Parquet mit Fallback
OUT_GZ_PKL = "graph_final_core_medium.pkl.gz"    # Fallback-Ziel
OUT_CHARGING_NODES = "charging_nodes_medium.parquet"
OUT_CHARGING_HUBS = "charging_hubs.medium.parquet"

gdf_pop_all = gpd.read_file(POP_GPKG)

print("CRS:", gdf_pop_all.crs)
print("Anzahl Zeilen:", len(gdf_pop_all))
print("Bounds:", gdf_pop_all.total_bounds)
print(gdf_pop_all.columns)

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def map_charging_to_nodes(node_coords, charging_csv_path, min_power_kw=150, max_snap_m=150):
    node_ids = np.array(list(node_coords.keys()))
    node_lats = np.array([node_coords[n][0] for n in node_ids])
    node_lons = np.array([node_coords[n][1] for n in node_ids])

    charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)
    charging_df.columns = charging_df.columns.str.strip()

    charging_df = charging_df[[
        "Ladeeinrichtungs-ID",
        "Breitengrad",
        "Längengrad",
        "Nennleistung Ladeeinrichtung [kW]"
    ]].dropna().copy()

    charging_df.columns = ["id", "lat", "lon", "power_kw"]

    charging_df["lat"] = charging_df["lat"].astype(str).str.replace(",", ".").astype(float)
    charging_df["lon"] = charging_df["lon"].astype(str).str.replace(",", ".").astype(float)
    charging_df["power_kw"] = charging_df["power_kw"].astype(str).str.replace(",", ".").astype(float)

    charging_df = charging_df[charging_df["power_kw"] >= min_power_kw].copy()

    print(f"Schnelllader >= {min_power_kw} kW: {len(charging_df):,}")

    charging_nodes = []

    for _, row in charging_df.iterrows():
        dists = haversine_vectorized(row["lat"], row["lon"], node_lats, node_lons)
        idx = np.argmin(dists)

        node_id = int(node_ids[idx])
        snap_dist = float(dists[idx])

        charging_nodes.append({
            "charger_id": row["id"],
            "node": node_id,
            "lat": row["lat"],
            "lon": row["lon"],
            "power_kw": row["power_kw"],
            "snap_dist_m": snap_dist
        })

    charging_nodes_df = pd.DataFrame(charging_nodes)
    charging_nodes_df = charging_nodes_df[charging_nodes_df["snap_dist_m"] <= max_snap_m].copy()

    print(f"Nach Snap-Filter <= {max_snap_m} m: {len(charging_nodes_df):,}")

    charging_hubs_df = charging_nodes_df.groupby("node").agg(
        max_power_kw=("power_kw", "max"),
        n_chargers=("charger_id", "count"),
        min_snap_dist_m=("snap_dist_m", "min")
    ).reset_index()

    print(f"Einzigartige Ladeknoten: {len(charging_hubs_df):,}")

    return charging_nodes_df, charging_hubs_df

def build_temp_roads_geometry(edges_df, node_coords):
    t0=time.time()
    rows=[]
    for r in edges_df[["arc_id","u","v"]].itertuples(index=False):
        c1=node_coords.get(r.u); c2=node_coords.get(r.v)
        if c1 and c2:
            rows.append((r.arc_id, LineString([(c1[1],c1[0]), (c2[1],c2[0])])))
    gdf=gpd.GeoDataFrame(rows, columns=["arc_id","geometry"], crs="EPSG:4326")
    print(f"Temp roads geometry: {len(gdf):,} | {time.time()-t0:.1f}s")
    return gdf

def map_population(edges_gz_pkl, pop_gpkg, buffer_m=50):
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={buffer_m}m")
    t_all=time.time()

    # 1) Edges aus gz-Pickle laden
    t0=time.time()
    with gzip.open(edges_gz_pkl,"rb") as f:
        edges = pd.DataFrame(pkl.load(f))
    print(f"read edges (gz-pkl) {time.time()-t0:.1f}s  | {len(edges):,} rows")

    # 2) Node-Coords laden
    with gzip.open(NODE_COORDS_PKL_GZ,"rb") as f:
        node_coords=pkl.load(f)

    # 3) temporäre Linien für Overlay
    gdf_roads = build_temp_roads_geometry(edges, node_coords).to_crs(3035)
    print(
    "Roads:",
    len(gdf_roads)
)
    gdf_roads["geom_buf"] = gdf_roads.geometry.buffer(buffer_m)
    gdf_roads = gdf_roads.set_geometry("geom_buf")

    # 4) Bevölkerung lesen + BBox-Filter in EPSG:4326
    t0=time.time()
    gdf_pop_all = gpd.read_file(pop_gpkg)[["T","geometry"]].rename(
    columns={"T":"population"}
    )

    gdf_pop_all = gdf_pop_all[
        gdf_pop_all["population"] > 0
    ]

    # zuerst nach WGS84 transformieren
    gdf_pop_all = gdf_pop_all.to_crs(4326)

    # dann BBox-Filter
    gdf_pop_all = gdf_pop_all.cx[
        LON_MIN:LON_MAX,
        LAT_MIN:LAT_MAX
    ].copy()

    # anschließend zurück nach 3035
    gdf_pop = gdf_pop_all.to_crs(3035)

    print(
    "Pop cells:",
    len(gdf_pop)
)
    gdf_pop["cell_area"]=gdf_pop.geometry.area
    print(f"pop cells after BBox: {len(gdf_pop):,}  | load+filter {time.time()-t0:.1f}s")

    # 5) Overlay
    t0=time.time()
    ov=gpd.overlay(gdf_roads, gdf_pop, how="intersection")
    print(f"overlay rows: {len(ov):,} | {time.time()-t0:.1f}s")

    if len(ov)>0:
        ov["overlap_area"]=ov.geometry.area
        ov["assigned_pop"]= ov["population"]*(ov["overlap_area"]/ov["cell_area"])
        pop_agg = ov.groupby("arc_id").agg(total_pop=("assigned_pop","sum")).reset_index()
    else:
        pop_agg = pd.DataFrame(columns=["arc_id","total_pop"])

    # 6) Merge + Scores
    out = edges.merge(pop_agg, on="arc_id", how="left")

    out["total_pop"] = (
        out["total_pop"]
        .fillna(0)
        .astype("float32")
    )

    out["pop_per_m"] = (
        out["total_pop"] /
        (out["length_m"] + 1)
    ).astype("float32")

    m = out["pop_per_m"].max()

    if m > 0:
        out["pop_score_norm"] = (
            out["pop_per_m"] / m
        ).astype("float32")
    else:
        out["pop_score_norm"] = out["pop_per_m"]

    # ==========================================================
    # 7) Datentypen
    # ==========================================================

    int_cols = {
        "arc_id": "int64",
        "u": "int64",
        "v": "int64",
        "highway": "int16",
    }

    if "tunnel" in out.columns:
        out["tunnel"] = out["tunnel"].astype("category")

    for col, dtype in int_cols.items():
        if col in out.columns:
            out[col] = out[col].astype(dtype)

    float_cols = [
        "dist",
        "length_m",
        "accidents",
        "weighted_accidents",
        "accident_score_norm",
        "total_pop",
        "pop_per_m",
        "pop_score_norm",
    ]

    for col in float_cols:
        if col in out.columns:
            out[col] = out[col].astype("float32")

    # Hilfsspalte entfernen
    if "accident_score" in out.columns:
        out = out.drop(columns=["accident_score"])

    # ==========================================================
    # DEBUG
    # ==========================================================

    print("\n===== ID CHECK =====")
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    print("\n===== DTYPES =====")
    print(out[["arc_id", "u", "v", "highway"]].dtypes)

    print("\n===== POPULATION =====")
    print("total_pop max:", out["total_pop"].max())
    print("pop_per_m max:", out["pop_per_m"].max())
    print("pop_score_norm max:", out["pop_score_norm"].max())

    print(f"\nmap_population: total {time.time()-t_all:.1f}s")

    return out


if __name__=="__main__":
    merged = map_population(ACCIDENTS_GZ_PKL, POP_GPKG, buffer_m=50)
    with gzip.open(NODE_COORDS_PKL_GZ, "rb") as f:
        node_coords = pkl.load(f)

    charging_nodes_df, charging_hubs_df = map_charging_to_nodes(
        node_coords,
        CHARGING_CSV,
        min_power_kw=150,
        max_snap_m=150
    )

    charging_nodes_df.to_parquet(
        OUT_CHARGING_NODES,
        compression="zstd",
        index=False
    )

    charging_hubs_df.to_parquet(
        OUT_CHARGING_HUBS,
        compression="zstd",
        index=False
    )

    print("Saved:", OUT_CHARGING_NODES)
    print("Saved:", OUT_CHARGING_HUBS)

    # Parquet schreiben (schnell/kompakt); bei Engine-Problemen Fallback auf gz-Pickle
    try:
        merged.to_parquet(OUT_PARQUET, compression="zstd", index=False)
        print("Saved:", OUT_PARQUET)
    except Exception as e:
        print("Parquet write failed, fallback to gz-Pickle:", e)
        with gzip.open(OUT_GZ_PKL,"wb") as f:
            pkl.dump(merged.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
        print("Saved:", OUT_GZ_PKL)


CRS: EPSG:3035
Anzahl Zeilen: 4595749
Bounds: [ 943000.  941000. 6505000. 5416000.]
Index(['GRD_ID', 'T', 'M', 'F', 'Y_LT15', 'Y_1564', 'Y_GE65', 'EMP', 'NAT',
       'EU_OTH', 'OTH', 'SAME', 'CHG_IN', 'CHG_OUT', 'LAND_SURFACE',
       'POPULATED', 'T_CI', 'M_CI', 'F_CI', 'Y_LT15_CI', 'Y_1564_CI',
       'Y_GE65_CI', 'EMP_CI', 'NAT_CI', 'EU_OTH_CI', 'OTH_CI', 'SAME_CI',
       'CHG_IN_CI', 'CHG_OUT_CI', 'geometry'],
      dtype='str')
BBox lat[50.105,52.302] lon[10.592,14.564] | PAD=0.2 | buffer=50m
read edges (gz-pkl) 4.0s  | 1,070,450 rows
Temp roads geometry: 1,070,450 | 10.4s
Roads: 1070450
Pop cells: 34093
pop cells after BBox: 34,093  | load+filter 44.8s
overlay rows: 1,109,017 | 52.5s

===== ID CHECK =====
u min: 442759
u max: 13868515003
v min: 442759
v max: 13868515003

===== DTYPES =====
arc_id     int64
u          int64
v          int64
highway    int16
dtype: object

===== POPULATION =====
total_pop max: 494.04105
pop_per_m max: 85.1737
pop_score_norm max: 1.0

map_populati

C:\Users\9631\AppData\Local\Temp\ipykernel_9628\1743947305.py:73: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)


Schnelllader >= 150 kW: 21,657
Nach Snap-Filter <= 150 m: 1,528
Einzigartige Ladeknoten: 627
Saved: charging_nodes_medium.parquet
Saved: charging_hubs.medium.parquet
Saved: graph_final_core_medium.parquet


In [ ]:
# 04_dijkstra_matrix.py — robust (Parquet → gz-Pickle Fallback) + sicherer nearest_nodes
import time, math, gzip, pickle as pkl
import pandas as pd
import numpy as np
import networkx as nx
 
# ===== I/O: finales Kantenformat aus 03 (Parquet ODER gz-Pickle) =====
# Wenn du in 03 Parquet geschrieben hast: Pfad auf *.parquet setzen.
# Wenn 03 auf gz-Pickle gefallen ist: Pfad auf *.pkl.gz setzen — der Reader unten kann beides.
GRAPH_FILE = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_medium.parquet"  # Parquet oder gz-Pickle
NODE_COORDS_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_medium.pkl.gz"
 
POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.367289, "lon": 12.434819, "type": "customer"},
    {"id": "C2",    "lat": 51.483631, "lon": 12.015296, "type": "customer"},
    {"id": "C3",    "lat": 51.084678, "lon": 13.768165, "type": "customer"},
    {"id": "C4",    "lat": 50.798920, "lon": 12.929745, "type": "customer"},
    {"id": "C5",    "lat": 50.979200, "lon": 11.125600, "type": "customer"},
    {"id": "C6",    "lat": 50.945592, "lon": 11.608864, "type": "customer"},
    {"id": "C7",    "lat": 50.961200, "lon": 11.259260, "type": "customer"},
    {"id": "C8",    "lat": 50.912492, "lon": 12.061403, "type": "customer"},
    {"id": "C9",    "lat": 52.101822, "lon": 11.630092, "type": "customer"},
    {"id": "C10",   "lat": 51.802704, "lon": 12.235987, "type": "customer"},
    {"id": "C11",   "lat": 51.629473, "lon": 12.301097, "type": "customer"},
    {"id": "C12",   "lat": 51.769678, "lon": 14.364260, "type": "customer"},
    {"id": "C13",   "lat": 50.709001, "lon": 12.481034, "type": "customer"},
    {"id": "C14",   "lat": 50.499501, "lon": 12.147448, "type": "customer"},
    {"id": "C15",   "lat": 50.304638, "lon": 11.919935, "type": "customer"},
    {"id": "C16",   "lat": 51.486191, "lon": 10.791956, "type": "customer"},
    {"id": "C17",   "lat": 51.884395, "lon": 11.078309, "type": "customer"},
    {"id": "C18",   "lat": 51.876171, "lon": 12.591305, "type": "customer"},
    {"id": "C19",   "lat": 51.552248, "lon": 12.970863, "type": "customer"},
    {"id": "C20",   "lat": 51.146041, "lon": 11.832647, "type": "customer"},
]

 
LOAD_STATES = {
    "loaded": True,
    "empty": False,
}
 
# ===== Profile / Gewichte =====
PROFILES = {
    "shortest": {
        "mode": "distance"
    },
 
    "safest": {
        "mode": "risk"
    },
 
    "fastest": {
        "mode": "time"
    }
}
 
W_ACC = 0.5
W_POP = 0.5
 
SPEED_KMH = {
    "motorway":80,"motorway_link":50,"trunk":70,"trunk_link":45,"primary":60,"primary_link":40,
    "secondary":50,"secondary_link":35,"tertiary":40,"tertiary_link":30,"residential":25,"unclassified":30
}
ROAD_PENALTY = {
    "motorway":0.0,"motorway_link":0.05,"trunk":0.08,"trunk_link":0.12,"primary":0.20,"primary_link":0.25,
    "secondary":0.35,"secondary_link":0.40,"tertiary":0.55,"tertiary_link":0.60,"residential":0.90,"unclassified":0.70
}
HIGHWAY_REV = {
    1:"motorway",2:"motorway_link",3:"trunk",4:"trunk_link",5:"primary",6:"primary_link",7:"secondary",
    8:"secondary_link",9:"tertiary",10:"tertiary_link",11:"residential",12:"unclassified"
}
 
def load_graph():
    t0 = time.time()
    # Versuche Parquet; wenn fehlschlägt (oder Datei ist pkl.gz): Fallback
    try:
        edges = pd.read_parquet(GRAPH_FILE)
        print(f"read graph (parquet): {len(edges):,} rows")
    except Exception as e:
        print("Parquet read failed or not a parquet file, fallback to gz-Pickle:", e)
        with gzip.open(GRAPH_FILE, "rb") as f:
            edges = pd.DataFrame(pkl.load(f))
        print(f"read graph (gz-pkl): {len(edges):,} rows")
 
    with gzip.open(NODE_COORDS_PKL,"rb") as f:
        node_coords = pkl.load(f)
    print(f"node_coords: {len(node_coords):,} | load {time.time()-t0:.1f}s")
 
    G = nx.DiGraph()
    for e in edges.itertuples(index=False):
        u = int(e.u); v = int(e.v)
        dist_km = float(e.dist) / 1000.0
        hwy = HIGHWAY_REV.get(int(e.highway), "unclassified")
 
        sp = SPEED_KMH.get(hwy, 30)
        time_h = dist_km / sp if sp > 0 else 999.0
 
        acc = float(getattr(e, "accident_score_norm", 0.0))
        pop = float(getattr(e, "pop_score_norm", 0.0))
        base = W_ACC*acc + W_POP*pop
        risk = base * dist_km
 
        road = ROAD_PENALTY.get(hwy, 0.7) * dist_km
 
        G.add_edge(u, v, dist_km=dist_km, time_h=time_h,
                   risk_cost=risk, road_penalty_cost=road, highway=hwy, tunnel=getattr(e, "tunnel", "no"))
 
    print(f"nx graph: nodes={G.number_of_nodes():,} edges={G.number_of_edges():,}")
    return G, node_coords
 
def nearest_nodes(points, node_coords, valid):
    # Nur Knoten verwenden, die sowohl in valid als auch in node_coords enthalten sind
    nodes = [n for n in valid if n in node_coords]
    if not nodes:
        raise RuntimeError("Keine gültigen Knoten mit Koordinaten nach Komponentenreduktion gefunden.")
    coords = np.array([node_coords[n] for n in nodes], dtype=float)  # (N, 2) = (lat, lon)
 
    out = []
    for p in points:
        lat, lon = float(p["lat"]), float(p["lon"])
        # euklidischer Proxy in (lat,lon) — für DE ok
        d = np.sum((coords - np.array([lat, lon]))**2, axis=1)
        idx = int(np.argmin(d))
        n = nodes[idx]
        out.append({**p, "node": n})
    return pd.DataFrame(out)
 
def make_weight(profile, loaded=True):
 
    mode = PROFILES[profile]["mode"]
 
    def w(u, v, d):
 
        # Tunnelverbot für beladenes Fahrzeug
        if loaded and d.get("tunnel", "no") != "no":
            return float("inf")
 
        if mode == "distance":
            return d["dist_km"]
 
        elif mode == "time":
            return d["time_h"]
 
        elif mode == "risk":
            return d["risk_cost"]
 
        return d["dist_km"]
 
    return w
 
def compute(G, s, t, prof, loaded=True):
    try:
        cost, path = nx.single_source_dijkstra(
            G, s, t,
            weight=make_weight(prof, loaded=loaded)
        )
 
        real_dist_km = sum(
            G[path[i]][path[i+1]]["dist_km"]
            for i in range(len(path)-1)
        )
 
        time_min = sum(
            G[path[i]][path[i+1]]["time_h"]
            for i in range(len(path)-1)
        ) * 60.0
 
        
        risk_total = sum(
            G[path[i]][path[i+1]]["risk_cost"]
            for i in range(len(path)-1)
        )
 
        road_penalty_total = sum(
            G[path[i]][path[i+1]]["road_penalty_cost"]
            for i in range(len(path)-1)
        )
 
        tunnel_used = False
 
        for i in range(len(path)-1):
 
            if G[path[i]][path[i+1]].get("tunnel", "no") != "no":
                tunnel_used = True
 
        return (
            True,
            real_dist_km,
            time_min,
            risk_total,
            road_penalty_total,
            cost,
            path,
            tunnel_used
        )
 
    except Exception:
        return (
            False,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            [],
            False
        )
 
if __name__ == "__main__":
    t_all = time.time()
 
    G, node_coords = load_graph()
 
    # ggf. auf größte Komponente reduzieren
    if not nx.is_weakly_connected(G):
        largest = max(nx.weakly_connected_components(G), key=len)
        G = G.subgraph(largest).copy()
        print(f"weakly connected -> reduced to nodes={len(G.nodes()):,}, edges={len(G.edges()):,}")
 
    valid = set(G.nodes())
 
    # optional: node_coords direkt auf gültige Knoten beschränken (schneller/kleiner)
    node_coords = {k: v for k, v in node_coords.items() if k in valid}
 
    pts_df = nearest_nodes(POINTS, node_coords, valid)
    print("mapped points:", [(r["id"], r["node"]) for _, r in pts_df.iterrows()])
 
    rows = []
    t0 = time.time()
    for s in pts_df.itertuples():
 
        for t in pts_df.itertuples():
 
            if s.id == t.id:
                continue
 
            for prof in PROFILES:
 
                for load_name, loaded in LOAD_STATES.items():
 
                    ok, dkm, tmin, risk_total, road_penalty_total, cost, path, tunnel_used = compute(
                        G,
                        s.node,
                        t.node,
                        prof,
                        loaded=loaded
                    )
 
                    rows.append({
                        "from": s.id,
                        "to": t.id,
 
                        "profile": prof,
                        "load_state": load_name,
 
                        "dist_km": dkm,
                        "cost": cost,
                        "time_min": tmin,
                        "risk_total": risk_total,
                        "road_penalty_total": road_penalty_total,
 
                        "reachable": ok,
 
                        "path_len": len(path),
                        "path": path,
                        "tunnel_used": tunnel_used
                    })
    print(f"OD computed in {time.time()-t0:.1f}s  | rows={len(rows):,}")
 
    pd.DataFrame(rows).to_csv("od_matrix_medium.csv", index=False)
    pts_df.to_csv("mapped_points_clean.csv", index=False)
    with gzip.open("routes_clean.pkl.gz","wb") as f:
        pkl.dump(rows, f, protocol=pkl.HIGHEST_PROTOCOL)
 
    print("Saved od_matrix_medium.csv  | total", f"{time.time()-t_all:.1f}s")

read graph (parquet): 1,070,450 rows
node_coords: 608,056 | load 0.6s
nx graph: nodes=608,056 edges=1,070,450
weakly connected -> reduced to nodes=567,990, edges=995,387
mapped points: [('DEPOT', 3705572847), ('C1', 5755959188), ('C2', 390534904), ('C3', 12657075673), ('C4', 598171845), ('C5', 76996770), ('C6', 312509629), ('C7', 77167892), ('C8', 264785942), ('C9', 11304792076), ('C10', 12978687186), ('C11', 13096182768), ('C12', 405163790), ('C13', 4946143117), ('C14', 2508642937), ('C15', 60288144), ('C16', 304140041), ('C17', 266896718), ('C18', 10724403577), ('C19', 7103568741), ('C20', 10282356192)]
OD computed in 1966.0s  | rows=2,520
Saved od_matrix_medium.csv  | total 1983.1s


Large Instanz

In [ ]:
# 01_build_graph.py
import osmium, math, gzip, pickle as pkl, time
from collections import defaultdict

KEEP_HIGHWAY = {
    "motorway","motorway_link",
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link", "tertiary","tertiary_link",
}
HIGHWAY_MAP = {
    "motorway":1,"motorway_link":2,"trunk":3,"trunk_link":4,
    "primary":5,"primary_link":6,"secondary":7,"secondary_link":8,
    "tertiary":9,"tertiary_link":10
}

GLOBAL_HIGHWAYS = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link"
}

LOCAL_HIGHWAYS = {
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
}

# Kunden/Depot:
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
    ("C21",   51.342678, 11.988740),
    ("C22",   51.184292, 12.017954),
    ("C23",   51.411562, 12.172039),
    ("C24",   51.512967, 12.337281),
    ("C25",   51.358887, 12.766669),
    ("C26",   51.220925, 12.718451),
    ("C27",   51.129939, 12.512646),
    ("C28",   51.001292, 12.455299),
    ("C29",   51.159790, 13.517739),
    ("C30",   51.314591, 13.280383),
    ("C31",   50.921715, 13.392756),
    ("C32",   51.168095, 14.413691),
    ("C33",   51.430276, 14.289119),
    ("C34",   51.527821, 14.016959),
    ("C35",   51.513295, 11.579354),
    ("C36",   51.480816, 11.315494),
    ("C37",   51.769277, 11.483116),
    ("C38",   51.798561, 11.755918),
    ("C39",   51.746434, 12.002407),
    ("C40",   51.867664, 11.557193),
    ("C41",   51.850656, 10.787768),
    ("C42",   51.790323, 11.168981),
    ("C43",   51.385616, 10.848824),
    ("C44",   51.198762, 10.478537),
    ("C45",   50.927969, 10.715398),
    ("C46",   50.867329, 10.948139),
    ("C47",   50.697818, 10.937663),
    ("C48",   50.560132, 10.371962),
    ("C49",   50.243487, 10.967718),
    ("C50",   49.910244, 10.869676),
]
PAD = 0.20  

LATS = [p[1] for p in TEST_POINTS]
LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

def in_bbox(lat, lon): 
    return LAT_MIN<=lat<=LAT_MAX and LON_MIN<=lon<=LON_MAX

def near_test_point(lat, lon, radius_m=15000):

    for _, tlat, tlon in TEST_POINTS:

        if haversine(
            lat, lon,
            tlat, tlon
        ) <= radius_m:
            return True

    return False
def haversine(lat1, lon1, lat2, lon2):
    R=6371000.0
    p1,p2 = math.radians(lat1), math.radians(lat2)
    dphi=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

class NodeCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_count=defaultdict(int)
        self.wc=0
        self.t0=time.time()
    def way(self, w):
        self.wc+=1
        if self.wc % 500_000 == 0:
            dt=time.time()-self.t0
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 1 … {self.wc:,} Ways  |  {spd:,.0f} Ways/s  |  t={dt:,.1f}s")
        hwy=w.tags.get("highway")
        if hwy not in KEEP_HIGHWAY: 
            return
        try:
            nodes = list(w.nodes)

            if len(nodes) < 2:
                return

            keep = False

            for n in nodes:

                if not n.location.valid():
                    continue

                lat, lon = n.location.lat, n.location.lon

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            if not keep:
                return

            for n in nodes:
                self.node_count[n.ref] += 1

        except:
            return

class Builder(osmium.SimpleHandler):
    def __init__(self,junc):
        super().__init__()
        self.j=junc; self.nc={}; self.edges=[]
        self.aid=0; self.wc=0; self.t0=time.time(); self.last_edges=0
    def way(self, w):
        self.wc+=1
        if self.wc % 200_000 == 0:
            dt=time.time()-self.t0
            de=self.edges.__len__()-self.last_edges
            spd=self.wc/max(dt,1e-9)
            print(f"Pass 2 … {self.wc:,} Ways | Edges: {len(self.edges):,} | ΔEdges:{de:,} | {spd:,.0f} Ways/s | t={dt:,.1f}s")
            self.last_edges=len(self.edges)
        hwy=w.tags.get("highway")
        tunnel = w.tags.get("tunnel", "no")
        if hwy not in KEEP_HIGHWAY: 
            return
        nodes=list(w.nodes)
        if len(nodes)<2: return
        coords={}; keep=False
        for n in nodes:

            try:

                lat, lon = n.location.lat, n.location.lon

                coords[n.ref] = (lat, lon)

                if hwy in GLOBAL_HIGHWAYS:

                    if in_bbox(lat, lon):
                        keep = True

                elif hwy in LOCAL_HIGHWAYS:

                    if near_test_point(
                        lat,
                        lon,
                        radius_m=15000
                    ):
                        keep = True

            except:
                return
        if not keep: return
        seg_start=nodes[0].ref; seg_dist=0.0
        for i in range(len(nodes)-1):
            n1,n2=nodes[i].ref,nodes[i+1].ref
            lat1,lon1=coords[n1]; lat2,lon2=coords[n2]
            seg_dist += haversine(lat1,lon1,lat2,lon2)
            is_jun = (n2 in self.j) or (i==len(nodes)-2)
            if is_jun:
                u,v=seg_start,n2
                self.nc[u]=coords[u]; self.nc[v]=coords[v]
                hcode=HIGHWAY_MAP.get(hwy,12)
                self.edges.append({
                    "arc_id": self.aid,
                    "u": u,
                    "v": v,
                    "dist": seg_dist,
                    "highway": hcode,
                    "tunnel": tunnel
                })
                self.aid += 1
                if w.tags.get("oneway","no") not in ("yes","1","true"):
                    self.edges.append({
                        "arc_id": self.aid,
                        "u": v,
                        "v": u,
                        "dist": seg_dist,
                        "highway": hcode,
                        "tunnel": tunnel
                    })
                    self.aid += 1
                seg_start=n2; seg_dist=0.0

def build(pbf_path, out_path="graph_core.pkl.gz"):
    print("=== BUILD GRAPH ===")
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}]  lon[{LON_MIN:.3f},{LON_MAX:.3f}]  | PAD={PAD}")
    t0=time.time()
    print("Pass 1 … counting junctions")
    c=NodeCounter(); c.apply_file(pbf_path, locations=True)
    t1=time.time()
    junc={n for n,k in c.node_count.items() if k!=2}
    print(f"Pass 1 done: {t1-t0:,.1f}s  | Junctions: {len(junc):,} (unique nodes: {len(c.node_count):,})")
    print("Pass 2 … building edges")
    b=Builder(junc); b.apply_file(pbf_path, locations=True)
    t2=time.time()
    core={"node_coords":b.nc,"arcs":b.edges}
    print(f"Pass 2 done: {t2-t1:,.1f}s  | Nodes: {len(b.nc):,} | Edges: {len(b.edges):,}")
    print(
    f"Edges/Node: "
    f"{len(b.edges)/max(len(b.nc),1):.2f}"
)
    with gzip.open(out_path,"wb") as f: pkl.dump(core,f,protocol=pkl.HIGHEST_PROTOCOL)
    print(f"Saved: {out_path}")
    print(f"Total time: {time.time()-t0:,.1f}s")

if __name__=="__main__":
    build(r"C:\Users\9631\Downloads\germany-latest.osm(1).pbf", "graph_core_large.pkl.gz")


=== BUILD GRAPH ===
BBox lat[49.710,52.302]  lon[10.172,14.614]  | PAD=0.2
Pass 1 … counting junctions
Pass 1 … 500,000 Ways  |  9,672 Ways/s  |  t=51.7s
Pass 1 … 1,000,000 Ways  |  12,676 Ways/s  |  t=78.9s
Pass 1 … 1,500,000 Ways  |  15,482 Ways/s  |  t=96.9s
Pass 1 … 2,000,000 Ways  |  17,728 Ways/s  |  t=112.8s
Pass 1 … 2,500,000 Ways  |  19,930 Ways/s  |  t=125.4s
Pass 1 … 3,000,000 Ways  |  21,724 Ways/s  |  t=138.1s
Pass 1 … 3,500,000 Ways  |  23,317 Ways/s  |  t=150.1s
Pass 1 … 4,000,000 Ways  |  25,079 Ways/s  |  t=159.5s
Pass 1 … 4,500,000 Ways  |  26,841 Ways/s  |  t=167.7s
Pass 1 … 5,000,000 Ways  |  28,462 Ways/s  |  t=175.7s
Pass 1 … 5,500,000 Ways  |  29,832 Ways/s  |  t=184.4s
Pass 1 … 6,000,000 Ways  |  31,192 Ways/s  |  t=192.4s
Pass 1 … 6,500,000 Ways  |  33,170 Ways/s  |  t=196.0s
Pass 1 … 7,000,000 Ways  |  34,993 Ways/s  |  t=200.0s
Pass 1 … 7,500,000 Ways  |  36,742 Ways/s  |  t=204.1s
Pass 1 … 8,000,000 Ways  |  38,281 Ways/s  |  t=209.0s
Pass 1 … 8,500,000 Ways

In [ ]:
# 02_map_accidents.py  —  gz-Pickle Variante (kein pyarrow nötig)
import gzip, pickle as pkl, time
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree

# ===== A) Punkte aus 01 (für BBox) =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
    ("C21",   51.342678, 11.988740),
    ("C22",   51.184292, 12.017954),
    ("C23",   51.411562, 12.172039),
    ("C24",   51.512967, 12.337281),
    ("C25",   51.358887, 12.766669),
    ("C26",   51.220925, 12.718451),
    ("C27",   51.129939, 12.512646),
    ("C28",   51.001292, 12.455299),
    ("C29",   51.159790, 13.517739),
    ("C30",   51.314591, 13.280383),
    ("C31",   50.921715, 13.392756),
    ("C32",   51.168095, 14.413691),
    ("C33",   51.430276, 14.289119),
    ("C34",   51.527821, 14.016959),
    ("C35",   51.513295, 11.579354),
    ("C36",   51.480816, 11.315494),
    ("C37",   51.769277, 11.483116),
    ("C38",   51.798561, 11.755918),
    ("C39",   51.746434, 12.002407),
    ("C40",   51.867664, 11.557193),
    ("C41",   51.850656, 10.787768),
    ("C42",   51.790323, 11.168981),
    ("C43",   51.385616, 10.848824),
    ("C44",   51.198762, 10.478537),
    ("C45",   50.927969, 10.715398),
    ("C46",   50.867329, 10.948139),
    ("C47",   50.697818, 10.937663),
    ("C48",   50.560132, 10.371962),
    ("C49",   50.243487, 10.967718),
    ("C50",   49.910244, 10.869676),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

BUFFER_M = 30  

# ===== B) I/O-Pfade (anpassen) =====
CORE_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_core_large.pkl.gz"
ACCIDENTS_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Accidents.csv"
OUT_EDGES_PKL_GZ = "accidents_core_large.pkl.gz"
OUT_NODECOORDS_PKL_GZ = "node_coords_large.pkl.gz"

def load_core(path):
    with gzip.open(path,"rb") as f:
        return pkl.load(f)

def load_acc(csv_path):
    t0=time.time()
    df=pd.read_csv(csv_path,sep=";",low_memory=False)
    df.columns=df.columns.str.strip()
    for c in ["XGCSWGS84","YGCSWGS84"]:
        df[c]=pd.to_numeric(df[c].astype(str).str.replace(",","."), errors="coerce")
    df=df.dropna(subset=["XGCSWGS84","YGCSWGS84"])
    df["weight"]=df["UKATEGORIE"].map({1:5,2:3,3:1}).fillna(1)
    g=gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["XGCSWGS84"], df["YGCSWGS84"]), crs="EPSG:4326")
    g=g.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX].copy()
    print(f"Accidents: total after BBox = {len(g):,}  | load+filter {time.time()-t0:.1f}s")
    return g

def build_roads_gdf(core):
    from shapely.geometry import LineString
    rows=[]
    nc=core["node_coords"]; arcs=core["arcs"]
    t0=time.time()
    for i,e in enumerate(arcs,1):
        c1=nc.get(e["u"]); c2=nc.get(e["v"])
        if not c1 or not c2: 
            continue
        rows.append({"arc_id":e["arc_id"],"u":e["u"],"v":e["v"],"dist":e["dist"],"highway":e["highway"], "tunnel": e.get("tunnel", "no"),
                     "geometry":LineString([(c1[1],c1[0]),(c2[1],c2[0])])})
        if i % 1_000_000 == 0:
            print(f"build_roads_gdf: {i:,}/{len(arcs):,} edges assembled")
    gdf=gpd.GeoDataFrame(rows, crs="EPSG:4326")
    print(f"Roads GDF: {len(gdf):,} | build {time.time()-t0:.1f}s")
    return gdf

def map_acc(roads, acc, buf_m=BUFFER_M):
    t0=time.time()
    roads_m=roads.to_crs(32632).copy()
    roads_m["length_m"]=roads_m.geometry.length
    geoms=list(roads_m.geometry.values)
    tree=STRtree(geoms)

    acc_m = acc.to_crs(32632)
    assigns=[]
    for i,(pt,w) in enumerate(zip(acc_m.geometry.values, acc_m["weight"].values),1):
        cands=tree.query(pt.buffer(buf_m))
        if len(cands) > 0:
            best=None; bd=1e9
            for ix in cands:
                ix_int = int(ix)
                d=pt.distance(geoms[ix_int])
                if d<bd:
                    bd=d; best=ix_int
            if (best is not None) and (bd<=buf_m):
                assigns.append((int(roads_m.iloc[best]["arc_id"]), float(w)))
        if i % 200_000 == 0:
            print(f"map_acc: processed {i:,} acc pts  | matches so far {len(assigns):,}")
    print(f"map_acc finished loop: {time.time()-t0:.1f}s | matches {len(assigns):,}")

    if not assigns:
        agg=pd.DataFrame(columns=["arc_id","accidents","weighted_accidents"])
    else:
        df=pd.DataFrame(assigns, columns=["arc_id","weight"])
        agg=df.groupby("arc_id").agg(accidents=("arc_id","count"),
                                     weighted_accidents=("weight","sum")).reset_index()

    out=roads_m[["arc_id","u","v","dist","highway", "tunnel","length_m"]].merge(agg, on="arc_id", how="left")
    out[["accidents","weighted_accidents"]]=out[["accidents","weighted_accidents"]].fillna(0)
    out["accident_score"]=out["weighted_accidents"]/(out["length_m"]+1)
    m=out["accident_score"].max()
    out["accident_score_norm"]= out["accident_score"]/m if m>0 else 0.0

    # kompakte Typen
    out = out.astype({
    "arc_id": "int64",
    "u": "int64",
    "v": "int64",
    "highway": "int16",
    "tunnel": "string",
    "dist": "float32",
    "length_m": "float32",

    "accidents": "float32",
    "weighted_accidents": "float32",

    "accident_score": "float32",
    "accident_score_norm": "float32",
})
    
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    return out

if __name__=="__main__":
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={BUFFER_M}m")
    t_all=time.time()
    t0=time.time(); core=load_core(CORE_PKL_GZ); print(f"load_core {time.time()-t0:.1f}s")
    t0=time.time(); roads=build_roads_gdf(core); print(f"build_roads_gdf {time.time()-t0:.1f}s")
    print("\nRoads GDF Speicherbedarf:")
    print(
        f"{roads.memory_usage(deep=True).sum()/1024**3:.2f} GB"
    )
    t0=time.time(); acc=load_acc(ACCIDENTS_CSV); print(f"load_acc {time.time()-t0:.1f}s")
    t0=time.time(); mapped=map_acc(roads,acc,buf_m=BUFFER_M); print(f"map_acc {time.time()-t0:.1f}s")

    # Speichern als gz-Pickle (robust, keine pyarrow-Abhängigkeit)
    with gzip.open(OUT_EDGES_PKL_GZ,"wb") as f:
        pkl.dump(mapped.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
    with gzip.open(OUT_NODECOORDS_PKL_GZ,"wb") as f:
        pkl.dump(core["node_coords"], f, protocol=pkl.HIGHEST_PROTOCOL)

    print(f"Saved: {OUT_EDGES_PKL_GZ}, {OUT_NODECOORDS_PKL_GZ}")
    print(f"Total {time.time()-t_all:.1f}s")


BBox lat[49.710,52.302] lon[10.172,14.614] | PAD=0.2 | buffer=30m
load_core 2.5s
build_roads_gdf: 1,000,000/1,719,736 edges assembled
Roads GDF: 1,719,736 | build 19.5s
build_roads_gdf 20.0s

Roads GDF Speicherbedarf:
0.09 GB
Accidents: total after BBox = 32,632  | load+filter 2.4s
load_acc 2.4s
map_acc finished loop: 9.9s | matches 19,550
u min: 442759
u max: 13868515003
v min: 442759
v max: 13868515003
map_acc 10.3s
Saved: accidents_core_large.pkl.gz, node_coords_large.pkl.gz
Total 52.6s


In [11]:
# 03_map_population_and_merge.py
import time, gzip, pickle as pkl
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import numpy as np

# ===== A) Punkte/BBox konsistent zu 01/02 =====
TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),
    ("C1",    51.367289, 12.434819),
    ("C2",    51.483631, 12.015296),
    ("C3",    51.084678, 13.768165),
    ("C4",    50.798920, 12.929745),
    ("C5",    50.979200, 11.125600),
    ("C6",    50.945592, 11.608864),
    ("C7",    50.961200, 11.259260),
    ("C8",    50.912492, 12.061403),
    ("C9",    52.101822, 11.630092),
    ("C10",   51.802704, 12.235987),
    ("C11",   51.629473, 12.301097),
    ("C12",   51.769678, 14.364260),
    ("C13",   50.709001, 12.481034),
    ("C14",   50.499501, 12.147448),
    ("C15",   50.304638, 11.919935),
    ("C16",   51.486191, 10.791956),
    ("C17",   51.884395, 11.078309),
    ("C18",   51.876171, 12.591305),
    ("C19",   51.552248, 12.970863),
    ("C20",   51.146041, 11.832647),
    ("C21",   51.342678, 11.988740),
    ("C22",   51.184292, 12.017954),
    ("C23",   51.411562, 12.172039),
    ("C24",   51.512967, 12.337281),
    ("C25",   51.358887, 12.766669),
    ("C26",   51.220925, 12.718451),
    ("C27",   51.129939, 12.512646),
    ("C28",   51.001292, 12.455299),
    ("C29",   51.159790, 13.517739),
    ("C30",   51.314591, 13.280383),
    ("C31",   50.921715, 13.392756),
    ("C32",   51.168095, 14.413691),
    ("C33",   51.430276, 14.289119),
    ("C34",   51.527821, 14.016959),
    ("C35",   51.513295, 11.579354),
    ("C36",   51.480816, 11.315494),
    ("C37",   51.769277, 11.483116),
    ("C38",   51.798561, 11.755918),
    ("C39",   51.746434, 12.002407),
    ("C40",   51.867664, 11.557193),
    ("C41",   51.850656, 10.787768),
    ("C42",   51.790323, 11.168981),
    ("C43",   51.385616, 10.848824),
    ("C44",   51.198762, 10.478537),
    ("C45",   50.927969, 10.715398),
    ("C46",   50.867329, 10.948139),
    ("C47",   50.697818, 10.937663),
    ("C48",   50.560132, 10.371962),
    ("C49",   50.243487, 10.967718),
    ("C50",   49.910244, 10.869676),
]
PAD = 0.20
LATS = [p[1] for p in TEST_POINTS]; LONS = [p[2] for p in TEST_POINTS]
LAT_MIN, LAT_MAX = min(LATS)-PAD, max(LATS)+PAD
LON_MIN, LON_MAX = min(LONS)-PAD, max(LONS)+PAD

# ===== B) I/O-Pfade =====
ACCIDENTS_GZ_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\accidents_core_large.pkl.gz"  # Output aus 02 (gz-Pickle)
NODE_COORDS_PKL_GZ = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_large.pkl.gz"   # aus 02
POP_GPKG = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\ESTAT_Census_2021_V2.gpkg"
CHARGING_CSV = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\Ladesaeulenregister_BNetzA_2026-04-22.csv"
OUT_PARQUET = "graph_final_core_large.parquet"  # Parquet mit Fallback
OUT_GZ_PKL = "graph_final_core_large.pkl.gz"    # Fallback-Ziel
OUT_CHARGING_NODES = "charging_nodes_large.parquet"
OUT_CHARGING_HUBS = "charging_hubs.large.parquet"

gdf_pop_all = gpd.read_file(POP_GPKG)

print("CRS:", gdf_pop_all.crs)
print("Anzahl Zeilen:", len(gdf_pop_all))
print("Bounds:", gdf_pop_all.total_bounds)
print(gdf_pop_all.columns)

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def map_charging_to_nodes(node_coords, charging_csv_path, min_power_kw=150, max_snap_m=150):
    node_ids = np.array(list(node_coords.keys()))
    node_lats = np.array([node_coords[n][0] for n in node_ids])
    node_lons = np.array([node_coords[n][1] for n in node_ids])

    charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)
    charging_df.columns = charging_df.columns.str.strip()

    charging_df = charging_df[[
        "Ladeeinrichtungs-ID",
        "Breitengrad",
        "Längengrad",
        "Nennleistung Ladeeinrichtung [kW]"
    ]].dropna().copy()

    charging_df.columns = ["id", "lat", "lon", "power_kw"]

    charging_df["lat"] = charging_df["lat"].astype(str).str.replace(",", ".").astype(float)
    charging_df["lon"] = charging_df["lon"].astype(str).str.replace(",", ".").astype(float)
    charging_df["power_kw"] = charging_df["power_kw"].astype(str).str.replace(",", ".").astype(float)

    charging_df = charging_df[charging_df["power_kw"] >= min_power_kw].copy()

    print(f"Schnelllader >= {min_power_kw} kW: {len(charging_df):,}")

    charging_nodes = []

    for _, row in charging_df.iterrows():
        dists = haversine_vectorized(row["lat"], row["lon"], node_lats, node_lons)
        idx = np.argmin(dists)

        node_id = int(node_ids[idx])
        snap_dist = float(dists[idx])

        charging_nodes.append({
            "charger_id": row["id"],
            "node": node_id,
            "lat": row["lat"],
            "lon": row["lon"],
            "power_kw": row["power_kw"],
            "snap_dist_m": snap_dist
        })

    charging_nodes_df = pd.DataFrame(charging_nodes)
    charging_nodes_df = charging_nodes_df[charging_nodes_df["snap_dist_m"] <= max_snap_m].copy()

    print(f"Nach Snap-Filter <= {max_snap_m} m: {len(charging_nodes_df):,}")

    charging_hubs_df = charging_nodes_df.groupby("node").agg(
        max_power_kw=("power_kw", "max"),
        n_chargers=("charger_id", "count"),
        min_snap_dist_m=("snap_dist_m", "min")
    ).reset_index()

    print(f"Einzigartige Ladeknoten: {len(charging_hubs_df):,}")

    return charging_nodes_df, charging_hubs_df

def build_temp_roads_geometry(edges_df, node_coords):
    t0=time.time()
    rows=[]
    for r in edges_df[["arc_id","u","v"]].itertuples(index=False):
        c1=node_coords.get(r.u); c2=node_coords.get(r.v)
        if c1 and c2:
            rows.append((r.arc_id, LineString([(c1[1],c1[0]), (c2[1],c2[0])])))
    gdf=gpd.GeoDataFrame(rows, columns=["arc_id","geometry"], crs="EPSG:4326")
    print(f"Temp roads geometry: {len(gdf):,} | {time.time()-t0:.1f}s")
    return gdf

def map_population(edges_gz_pkl, pop_gpkg, buffer_m=50):
    print(f"BBox lat[{LAT_MIN:.3f},{LAT_MAX:.3f}] lon[{LON_MIN:.3f},{LON_MAX:.3f}] | PAD={PAD} | buffer={buffer_m}m")
    t_all=time.time()

    # 1) Edges aus gz-Pickle laden
    t0=time.time()
    with gzip.open(edges_gz_pkl,"rb") as f:
        edges = pd.DataFrame(pkl.load(f))
    print(f"read edges (gz-pkl) {time.time()-t0:.1f}s  | {len(edges):,} rows")

    # 2) Node-Coords laden
    with gzip.open(NODE_COORDS_PKL_GZ,"rb") as f:
        node_coords=pkl.load(f)

    # 3) temporäre Linien für Overlay
    gdf_roads = build_temp_roads_geometry(edges, node_coords).to_crs(3035)
    print(
    "Roads:",
    len(gdf_roads)
)
    gdf_roads["geom_buf"] = gdf_roads.geometry.buffer(buffer_m)
    gdf_roads = gdf_roads.set_geometry("geom_buf")

    # 4) Bevölkerung lesen + BBox-Filter in EPSG:4326
    t0=time.time()
    gdf_pop_all = gpd.read_file(pop_gpkg)[["T","geometry"]].rename(
    columns={"T":"population"}
    )

    gdf_pop_all = gdf_pop_all[
        gdf_pop_all["population"] > 0
    ]

    # zuerst nach WGS84 transformieren
    gdf_pop_all = gdf_pop_all.to_crs(4326)

    # dann BBox-Filter
    gdf_pop_all = gdf_pop_all.cx[
        LON_MIN:LON_MAX,
        LAT_MIN:LAT_MAX
    ].copy()

    # anschließend zurück nach 3035
    gdf_pop = gdf_pop_all.to_crs(3035)

    print(
    "Pop cells:",
    len(gdf_pop)
)
    gdf_pop["cell_area"]=gdf_pop.geometry.area
    print(f"pop cells after BBox: {len(gdf_pop):,}  | load+filter {time.time()-t0:.1f}s")

    # 5) Overlay
    t0=time.time()
    ov=gpd.overlay(gdf_roads, gdf_pop, how="intersection")
    print(f"overlay rows: {len(ov):,} | {time.time()-t0:.1f}s")

    if len(ov)>0:
        ov["overlap_area"]=ov.geometry.area
        ov["assigned_pop"]= ov["population"]*(ov["overlap_area"]/ov["cell_area"])
        pop_agg = ov.groupby("arc_id").agg(total_pop=("assigned_pop","sum")).reset_index()
    else:
        pop_agg = pd.DataFrame(columns=["arc_id","total_pop"])

    # 6) Merge + Scores
    out = edges.merge(pop_agg, on="arc_id", how="left")

    out["total_pop"] = (
        out["total_pop"]
        .fillna(0)
        .astype("float32")
    )

    out["pop_per_m"] = (
        out["total_pop"] /
        (out["length_m"] + 1)
    ).astype("float32")

    m = out["pop_per_m"].max()

    if m > 0:
        out["pop_score_norm"] = (
            out["pop_per_m"] / m
        ).astype("float32")
    else:
        out["pop_score_norm"] = out["pop_per_m"]

    # ==========================================================
    # 7) Datentypen
    # ==========================================================

    int_cols = {
        "arc_id": "int64",
        "u": "int64",
        "v": "int64",
        "highway": "int16",
    }

    if "tunnel" in out.columns:
        out["tunnel"] = out["tunnel"].astype("category")

    for col, dtype in int_cols.items():
        if col in out.columns:
            out[col] = out[col].astype(dtype)

    float_cols = [
        "dist",
        "length_m",
        "accidents",
        "weighted_accidents",
        "accident_score_norm",
        "total_pop",
        "pop_per_m",
        "pop_score_norm",
    ]

    for col in float_cols:
        if col in out.columns:
            out[col] = out[col].astype("float32")

    # Hilfsspalte entfernen
    if "accident_score" in out.columns:
        out = out.drop(columns=["accident_score"])

    # ==========================================================
    # DEBUG
    # ==========================================================

    print("\n===== ID CHECK =====")
    print("u min:", out["u"].min())
    print("u max:", out["u"].max())
    print("v min:", out["v"].min())
    print("v max:", out["v"].max())

    print("\n===== DTYPES =====")
    print(out[["arc_id", "u", "v", "highway"]].dtypes)

    print("\n===== POPULATION =====")
    print("total_pop max:", out["total_pop"].max())
    print("pop_per_m max:", out["pop_per_m"].max())
    print("pop_score_norm max:", out["pop_score_norm"].max())

    print(f"\nmap_population: total {time.time()-t_all:.1f}s")

    return out


if __name__=="__main__":
    merged = map_population(ACCIDENTS_GZ_PKL, POP_GPKG, buffer_m=50)
    with gzip.open(NODE_COORDS_PKL_GZ, "rb") as f:
        node_coords = pkl.load(f)

    charging_nodes_df, charging_hubs_df = map_charging_to_nodes(
        node_coords,
        CHARGING_CSV,
        min_power_kw=150,
        max_snap_m=150
    )

    charging_nodes_df.to_parquet(
        OUT_CHARGING_NODES,
        compression="zstd",
        index=False
    )

    charging_hubs_df.to_parquet(
        OUT_CHARGING_HUBS,
        compression="zstd",
        index=False
    )

    print("Saved:", OUT_CHARGING_NODES)
    print("Saved:", OUT_CHARGING_HUBS)

    # Parquet schreiben (schnell/kompakt); bei Engine-Problemen Fallback auf gz-Pickle
    try:
        merged.to_parquet(OUT_PARQUET, compression="zstd", index=False)
        print("Saved:", OUT_PARQUET)
    except Exception as e:
        print("Parquet write failed, fallback to gz-Pickle:", e)
        with gzip.open(OUT_GZ_PKL,"wb") as f:
            pkl.dump(merged.to_dict("records"), f, protocol=pkl.HIGHEST_PROTOCOL)
        print("Saved:", OUT_GZ_PKL)


CRS: EPSG:3035
Anzahl Zeilen: 4595749
Bounds: [ 943000.  941000. 6505000. 5416000.]
Index(['GRD_ID', 'T', 'M', 'F', 'Y_LT15', 'Y_1564', 'Y_GE65', 'EMP', 'NAT',
       'EU_OTH', 'OTH', 'SAME', 'CHG_IN', 'CHG_OUT', 'LAND_SURFACE',
       'POPULATED', 'T_CI', 'M_CI', 'F_CI', 'Y_LT15_CI', 'Y_1564_CI',
       'Y_GE65_CI', 'EMP_CI', 'NAT_CI', 'EU_OTH_CI', 'OTH_CI', 'SAME_CI',
       'CHG_IN_CI', 'CHG_OUT_CI', 'geometry'],
      dtype='str')
BBox lat[49.710,52.302] lon[10.172,14.614] | PAD=0.2 | buffer=50m
read edges (gz-pkl) 7.4s  | 1,719,736 rows
Temp roads geometry: 1,719,736 | 16.8s
Roads: 1719736
Pop cells: 45325
pop cells after BBox: 45,325  | load+filter 48.9s
overlay rows: 1,749,957 | 91.9s

===== ID CHECK =====
u min: 442759
u max: 13868515003
v min: 442759
v max: 13868515003

===== DTYPES =====
arc_id     int64
u          int64
v          int64
highway    int16
dtype: object

===== POPULATION =====
total_pop max: 494.04105
pop_per_m max: 85.1737
pop_score_norm max: 1.0

map_populati

C:\Users\9631\AppData\Local\Temp\ipykernel_9628\502390366.py:103: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)


Schnelllader >= 150 kW: 21,657
Nach Snap-Filter <= 150 m: 2,034
Einzigartige Ladeknoten: 814
Saved: charging_nodes_large.parquet
Saved: charging_hubs.large.parquet
Saved: graph_final_core_large.parquet


In [ ]:
# 04_dijkstra_matrix.py — robust (Parquet → gz-Pickle Fallback) + sicherer nearest_nodes
import time, math, gzip, pickle as pkl
import pandas as pd
import numpy as np
import networkx as nx
 
# ===== I/O: finales Kantenformat aus 03 (Parquet ODER gz-Pickle) =====
# Wenn du in 03 Parquet geschrieben hast: Pfad auf *.parquet setzen.
# Wenn 03 auf gz-Pickle gefallen ist: Pfad auf *.pkl.gz setzen — der Reader unten kann beides.
GRAPH_FILE = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_large.parquet"  # Parquet oder gz-Pickle
NODE_COORDS_PKL = r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_large.pkl.gz"
 
POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.367289, "lon": 12.434819, "type": "customer"},
    {"id": "C2",    "lat": 51.483631, "lon": 12.015296, "type": "customer"},
    {"id": "C3",    "lat": 51.084678, "lon": 13.768165, "type": "customer"},
    {"id": "C4",    "lat": 50.798920, "lon": 12.929745, "type": "customer"},
    {"id": "C5",    "lat": 50.979200, "lon": 11.125600, "type": "customer"},
    {"id": "C6",    "lat": 50.945592, "lon": 11.608864, "type": "customer"},
    {"id": "C7",    "lat": 50.961200, "lon": 11.259260, "type": "customer"},
    {"id": "C8",    "lat": 50.912492, "lon": 12.061403, "type": "customer"},
    {"id": "C9",    "lat": 52.101822, "lon": 11.630092, "type": "customer"},
    {"id": "C10",   "lat": 51.802704, "lon": 12.235987, "type": "customer"},
    {"id": "C11",   "lat": 51.629473, "lon": 12.301097, "type": "customer"},
    {"id": "C12",   "lat": 51.769678, "lon": 14.364260, "type": "customer"},
    {"id": "C13",   "lat": 50.709001, "lon": 12.481034, "type": "customer"},
    {"id": "C14",   "lat": 50.499501, "lon": 12.147448, "type": "customer"},
    {"id": "C15",   "lat": 50.304638, "lon": 11.919935, "type": "customer"},
    {"id": "C16",   "lat": 51.486191, "lon": 10.791956, "type": "customer"},
    {"id": "C17",   "lat": 51.884395, "lon": 11.078309, "type": "customer"},
    {"id": "C18",   "lat": 51.876171, "lon": 12.591305, "type": "customer"},
    {"id": "C19",   "lat": 51.552248, "lon": 12.970863, "type": "customer"},
    {"id": "C20",   "lat": 51.146041, "lon": 11.832647, "type": "customer"},
    {"id": "C21",   "lat": 51.342678, "lon": 11.988740, "type": "customer"},
    {"id": "C22",   "lat": 51.184292, "lon": 12.017954, "type": "customer"},
    {"id": "C23",   "lat": 51.411562, "lon": 12.172039, "type": "customer"},
    {"id": "C24",   "lat": 51.512967, "lon": 12.337281, "type": "customer"},
    {"id": "C25",   "lat": 51.358887, "lon": 12.766669, "type": "customer"},
    {"id": "C26",   "lat": 51.220925, "lon": 12.718451, "type": "customer"},
    {"id": "C27",   "lat": 51.129939, "lon": 12.512646, "type": "customer"},
    {"id": "C28",   "lat": 51.001292, "lon": 12.455299, "type": "customer"},
    {"id": "C29",   "lat": 51.159790, "lon": 13.517739, "type": "customer"},
    {"id": "C30",   "lat": 51.314591, "lon": 13.280383, "type": "customer"},
    {"id": "C31",   "lat": 50.921715, "lon": 13.392756, "type": "customer"},
    {"id": "C32",   "lat": 51.168095, "lon": 14.413691, "type": "customer"},
    {"id": "C33",   "lat": 51.430276, "lon": 14.289119, "type": "customer"},
    {"id": "C34",   "lat": 51.527821, "lon": 14.016959, "type": "customer"},
    {"id": "C35",   "lat": 51.513295, "lon": 11.579354, "type": "customer"},
    {"id": "C36",   "lat": 51.480816, "lon": 11.315494, "type": "customer"},
    {"id": "C37",   "lat": 51.769277, "lon": 11.483116, "type": "customer"},
    {"id": "C38",   "lat": 51.798561, "lon": 11.755918, "type": "customer"},
    {"id": "C39",   "lat": 51.746434, "lon": 12.002407, "type": "customer"},
    {"id": "C40",   "lat": 51.867664, "lon": 11.557193, "type": "customer"},
    {"id": "C41",   "lat": 51.850656, "lon": 10.787768, "type": "customer"},
    {"id": "C42",   "lat": 51.790323, "lon": 11.168981, "type": "customer"},
    {"id": "C43",   "lat": 51.385616, "lon": 10.848824, "type": "customer"},
    {"id": "C44",   "lat": 51.198762, "lon": 10.478537, "type": "customer"},
    {"id": "C45",   "lat": 50.927969, "lon": 10.715398, "type": "customer"},
    {"id": "C46",   "lat": 50.867329, "lon": 10.948139, "type": "customer"},
    {"id": "C47",   "lat": 50.697818, "lon": 10.937663, "type": "customer"},
    {"id": "C48",   "lat": 50.560132, "lon": 10.371962, "type": "customer"},
    {"id": "C49",   "lat": 50.243487, "lon": 10.967718, "type": "customer"},
    {"id": "C50",   "lat": 49.910244, "lon": 10.869676, "type": "customer"},
]
 
LOAD_STATES = {
    "loaded": True,
    "empty": False,
}
 
# ===== Profile / Gewichte =====
PROFILES = {
    "shortest": {
        "mode": "distance"
    },
 
    "safest": {
        "mode": "risk"
    },
 
    "fastest": {
        "mode": "time"
    }
}
 
W_ACC = 0.5
W_POP = 0.5
 
SPEED_KMH = {
    "motorway":80,"motorway_link":50,"trunk":70,"trunk_link":45,"primary":60,"primary_link":40,
    "secondary":50,"secondary_link":35,"tertiary":40,"tertiary_link":30,"residential":25,"unclassified":30
}
ROAD_PENALTY = {
    "motorway":0.0,"motorway_link":0.05,"trunk":0.08,"trunk_link":0.12,"primary":0.20,"primary_link":0.25,
    "secondary":0.35,"secondary_link":0.40,"tertiary":0.55,"tertiary_link":0.60,"residential":0.90,"unclassified":0.70
}
HIGHWAY_REV = {
    1:"motorway",2:"motorway_link",3:"trunk",4:"trunk_link",5:"primary",6:"primary_link",7:"secondary",
    8:"secondary_link",9:"tertiary",10:"tertiary_link",11:"residential",12:"unclassified"
}
 
def load_graph():
    t0 = time.time()
    # Versuche Parquet; wenn fehlschlägt (oder Datei ist pkl.gz): Fallback
    try:
        edges = pd.read_parquet(GRAPH_FILE)
        print(f"read graph (parquet): {len(edges):,} rows")
    except Exception as e:
        print("Parquet read failed or not a parquet file, fallback to gz-Pickle:", e)
        with gzip.open(GRAPH_FILE, "rb") as f:
            edges = pd.DataFrame(pkl.load(f))
        print(f"read graph (gz-pkl): {len(edges):,} rows")
 
    with gzip.open(NODE_COORDS_PKL,"rb") as f:
        node_coords = pkl.load(f)
    print(f"node_coords: {len(node_coords):,} | load {time.time()-t0:.1f}s")
 
    G = nx.DiGraph()
    for e in edges.itertuples(index=False):
        u = int(e.u); v = int(e.v)
        dist_km = float(e.dist) / 1000.0
        hwy = HIGHWAY_REV.get(int(e.highway), "unclassified")
 
        sp = SPEED_KMH.get(hwy, 30)
        time_h = dist_km / sp if sp > 0 else 999.0
 
        acc = float(getattr(e, "accident_score_norm", 0.0))
        pop = float(getattr(e, "pop_score_norm", 0.0))
        base = W_ACC*acc + W_POP*pop
        risk = base * dist_km
 
        road = ROAD_PENALTY.get(hwy, 0.7) * dist_km
 
        G.add_edge(u, v, dist_km=dist_km, time_h=time_h,
                   risk_cost=risk, road_penalty_cost=road, highway=hwy, tunnel=getattr(e, "tunnel", "no"))
 
    print(f"nx graph: nodes={G.number_of_nodes():,} edges={G.number_of_edges():,}")
    return G, node_coords
 
def nearest_nodes(points, node_coords, valid):
    # Nur Knoten verwenden, die sowohl in valid als auch in node_coords enthalten sind
    nodes = [n for n in valid if n in node_coords]
    if not nodes:
        raise RuntimeError("Keine gültigen Knoten mit Koordinaten nach Komponentenreduktion gefunden.")
    coords = np.array([node_coords[n] for n in nodes], dtype=float)  # (N, 2) = (lat, lon)
 
    out = []
    for p in points:
        lat, lon = float(p["lat"]), float(p["lon"])
        # euklidischer Proxy in (lat,lon) — für DE ok
        d = np.sum((coords - np.array([lat, lon]))**2, axis=1)
        idx = int(np.argmin(d))
        n = nodes[idx]
        out.append({**p, "node": n})
    return pd.DataFrame(out)
 
def make_weight(profile, loaded=True):
 
    mode = PROFILES[profile]["mode"]
 
    def w(u, v, d):
 
        # Tunnelverbot für beladenes Fahrzeug
        if loaded and d.get("tunnel", "no") != "no":
            return float("inf")
 
        if mode == "distance":
            return d["dist_km"]
 
        elif mode == "time":
            return d["time_h"]
 
        elif mode == "risk":
            return d["risk_cost"]
 
        return d["dist_km"]
 
    return w
 
def compute(G, s, t, prof, loaded=True):
    try:
        cost, path = nx.single_source_dijkstra(
            G, s, t,
            weight=make_weight(prof, loaded=loaded)
        )
 
        real_dist_km = sum(
            G[path[i]][path[i+1]]["dist_km"]
            for i in range(len(path)-1)
        )
 
        time_min = sum(
            G[path[i]][path[i+1]]["time_h"]
            for i in range(len(path)-1)
        ) * 60.0
 
        
        risk_total = sum(
            G[path[i]][path[i+1]]["risk_cost"]
            for i in range(len(path)-1)
        )
 
        road_penalty_total = sum(
            G[path[i]][path[i+1]]["road_penalty_cost"]
            for i in range(len(path)-1)
        )
 
        tunnel_used = False
 
        for i in range(len(path)-1):
 
            if G[path[i]][path[i+1]].get("tunnel", "no") != "no":
                tunnel_used = True
 
        return (
            True,
            real_dist_km,
            time_min,
            risk_total,
            road_penalty_total,
            cost,
            path,
            tunnel_used
        )
 
    except Exception:
        return (
            False,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            [],
            False
        )
 
if __name__ == "__main__":
    t_all = time.time()
 
    G, node_coords = load_graph()
 
    # ggf. auf größte Komponente reduzieren
    if not nx.is_weakly_connected(G):
        largest = max(nx.weakly_connected_components(G), key=len)
        G = G.subgraph(largest).copy()
        print(f"weakly connected -> reduced to nodes={len(G.nodes()):,}, edges={len(G.edges()):,}")
 
    valid = set(G.nodes())
 
    # optional: node_coords direkt auf gültige Knoten beschränken (schneller/kleiner)
    node_coords = {k: v for k, v in node_coords.items() if k in valid}
 
    pts_df = nearest_nodes(POINTS, node_coords, valid)
    print("mapped points:", [(r["id"], r["node"]) for _, r in pts_df.iterrows()])
 
    rows = []
    t0 = time.time()
    for s in pts_df.itertuples():
 
        for t in pts_df.itertuples():
 
            if s.id == t.id:
                continue
 
            for prof in PROFILES:
 
                for load_name, loaded in LOAD_STATES.items():
 
                    ok, dkm, tmin, risk_total, road_penalty_total, cost, path, tunnel_used = compute(
                        G,
                        s.node,
                        t.node,
                        prof,
                        loaded=loaded
                    )
 
                    rows.append({
                        "from": s.id,
                        "to": t.id,
 
                        "profile": prof,
                        "load_state": load_name,
 
                        "dist_km": dkm,
                        "cost": cost,
                        "time_min": tmin,
                        "risk_total": risk_total,
                        "road_penalty_total": road_penalty_total,
 
                        "reachable": ok,
 
                        "path_len": len(path),
                        "path": path,
                        "tunnel_used": tunnel_used
                    })
    print(f"OD computed in {time.time()-t0:.1f}s  | rows={len(rows):,}")
 
    pd.DataFrame(rows).to_csv("od_matrix_large.csv", index=False)
    pts_df.to_csv("mapped_points_clean.csv", index=False)
    with gzip.open("routes_clean.pkl.gz","wb") as f:
        pkl.dump(rows, f, protocol=pkl.HIGHEST_PROTOCOL)
 
    print("Saved od_matrix_large.csv  | total", f"{time.time()-t_all:.1f}s")

read graph (parquet): 1,719,736 rows
node_coords: 953,326 | load 1.4s
nx graph: nodes=953,326 edges=1,719,736
weakly connected -> reduced to nodes=906,818, edges=1,632,577
mapped points: [('DEPOT', 3705572847), ('C1', 5755959188), ('C2', 390534904), ('C3', 12657075673), ('C4', 598171845), ('C5', 76996770), ('C6', 312509629), ('C7', 77167892), ('C8', 264785942), ('C9', 11304792076), ('C10', 12978687186), ('C11', 13096182768), ('C12', 405163790), ('C13', 4946143117), ('C14', 2508642937), ('C15', 60288144), ('C16', 304140041), ('C17', 266896718), ('C18', 10724403577), ('C19', 7103568741), ('C20', 10282356192), ('C21', 3101590538), ('C22', 1528179598), ('C23', 1928529456), ('C24', 10742853689), ('C25', 1032444130), ('C26', 1528123299), ('C27', 2906967066), ('C28', 279455411), ('C29', 628129650), ('C30', 130207645), ('C31', 1124818025), ('C32', 4955313228), ('C33', 4946317175), ('C34', 1608087139), ('C35', 313199536), ('C36', 249720548), ('C37', 3689235794), ('C38', 11456165420), ('C39', 29

Charging Matrix

Small 

In [ ]:

import pickle
import gzip
import pandas as pd
import numpy as np
import networkx as nx

# CONFIG

OUTPUT_CSV = "od_matrix_small_charger.csv"

BOUNDING_BUFFER_DEG = 1.0
MAX_CHARGERS_PER_CUSTOMER = 3


# POINTS 

POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.413933, "lon": 12.291324, "type": "customer"},
    {"id": "C2",    "lat": 51.487547, "lon": 12.078983, "type": "customer"},
    {"id": "C3",    "lat": 51.141971, "lon": 11.824094, "type": "customer"},
    {"id": "C4",    "lat": 51.208217, "lon": 11.960303, "type": "customer"},
    {"id": "C5",    "lat": 50.884000, "lon": 11.597000, "type": "customer"},
    {"id": "C6",    "lat": 50.912463, "lon": 12.061472, "type": "customer"},
    {"id": "C7",    "lat": 51.811731, "lon": 12.228786, "type": "customer"},
    {"id": "C8",    "lat": 51.009103, "lon": 12.462403, "type": "customer"},
    {"id": "C9",    "lat": 51.511895, "lon": 11.587541, "type": "customer"},
    {"id": "C10",   "lat": 51.055287, "lon": 12.110381, "type": "customer"},
]

points_df = pd.DataFrame(POINTS)


# PROFILE FUNKTION

def weight_func(profile):

    def w(u, v, d):

        if profile == "safest" and d.get("tunnel", "no") != "no":
            return float("inf")

        if profile == "shortest":
            return d["dist_km"]

        if profile == "fastest":
            return d["time_h"]

        if profile == "safest":
            return d["dist_km"] + 2.0 * d["risk"] + 0.5 * d["road_penalty"]

        return d["dist_km"]

    return w

# GRAPH LADEN

edges = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_small.parquet"
)

charging_hubs = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\charging_hubs.small.parquet"
)

with gzip.open(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_small.pkl.gz",
    "rb"
) as f:
    node_coords = pickle.load(f)

# GRAPH BUILD

G = nx.DiGraph()

for edge in edges.itertuples(index=False):

    u = int(edge.u)
    v = int(edge.v)

    dist_km = float(edge.dist) / 1000.0

    speed = 70
    time_h = dist_km / speed

    risk = (
        float(getattr(edge, "accident_score_norm", 0.0))
        * float(getattr(edge, "pop_score_norm", 0.0))
    )

    highway = int(getattr(edge, "highway", 12))

    if highway <= 6:
        road_penalty = 0.0
    else:
        road_penalty = 0.3

    tunnel = getattr(edge, "tunnel", "no")

    G.add_edge(
        u,
        v,
        dist_km=dist_km,
        time_h=time_h,
        risk=risk,
        road_penalty=road_penalty,
        tunnel=tunnel
    )


print(f"Graph: {G.number_of_nodes()} nodes")

# BOUNDING BOX

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# LADESAULEN REDUZIEREN

print("\nLadesäulen auswählen...")

charger_points = []

charging_hubs = charging_hubs.to_dict("records")

for i, ch in enumerate(charging_hubs):

    lat, lon = node_coords[ch["node"]]

    if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
        charger_points.append({
            "id": f"L{i}",
            "lat": lat,
            "lon": lon,
            "node": ch["node"],
            "type": "charger"
        })

print(f"Ladesäulen in Box: {len(charger_points)}")

selected_ids = set()

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (loc["lat"] - ch["lat"])**2 + (loc["lon"] - ch["lon"])**2
        dists.append((d, ch["id"]))

    dists.sort()

    nearest = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

    selected_ids.update(nearest)

charger_points = [ch for ch in charger_points if ch["id"] in selected_ids]

customer_to_chargers = {}

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (
            (loc["lat"] - ch["lat"])**2 +
            (loc["lon"] - ch["lon"])**2
        )

        dists.append((d, ch["id"]))

    dists.sort()

    customer_to_chargers[loc["id"]] = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

print(f"Reduzierte Ladesäulen: {len(charger_points)}")

points_df = pd.concat([points_df, pd.DataFrame(charger_points)], ignore_index=True)



# NEAREST NODE

node_ids = np.array(list(node_coords.keys()))
node_lats = np.array([node_coords[n][0] for n in node_ids])
node_lons = np.array([node_coords[n][1] for n in node_ids])

def nearest(lat, lon):
    d = (node_lats - lat)**2 + (node_lons - lon)**2
    return int(node_ids[np.argmin(d)])

if "node" not in points_df.columns:
    points_df["node"] = np.nan

points_df["node"] = points_df["node"].fillna(
    points_df.apply(
        lambda r: nearest(r.lat, r.lon),
        axis=1
    )
)

customers_df = points_df[
    points_df["type"].isin(["customer", "depot"])
].copy()

chargers_df = points_df[
    points_df["type"] == "charger"
].copy()

charger_lookup = {
    row["id"]: row
    for _, row in chargers_df.iterrows()
}

print(points_df["type"].value_counts())
print("\nBerechne OD Matrix...")

records = []
routes = {}


# CUSTOMER -> CHARGER

for _, src in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        print(f"{src.id} | {profile}")

        for charger_id in customer_to_chargers[src.id]:

            dst = charger_lookup[charger_id]

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type,
                "to_type": dst.type,
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })


# CHARGER -> CUSTOMER

for _, dst in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        allowed_chargers = customer_to_chargers[dst.id]

        for charger_id in allowed_chargers:

            src = charger_lookup[charger_id]

            print(f"{src.id} -> {dst.id} | {profile}")

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type if hasattr(src, "type") else "customer",
                "to_type": dst.type if hasattr(dst, "type") else "customer",
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
with open("routes_small_charger.pkl", "wb") as f:
    pickle.dump(
        routes,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("\n✅ Fertig!")
print("Matrixgröße:", len(df))


Graph: 232865 nodes

Ladesäulen auswählen...
Ladesäulen in Box: 212
Reduzierte Ladesäulen: 33
type
charger     33
customer    10
depot        1
Name: count, dtype: int64

Berechne OD Matrix...
DEPOT | shortest
DEPOT | fastest
DEPOT | safest
C1 | shortest
C1 | fastest
C1 | safest
C2 | shortest
C2 | fastest
C2 | safest
C3 | shortest
C3 | fastest
C3 | safest
C4 | shortest
C4 | fastest
C4 | safest
C5 | shortest
C5 | fastest
C5 | safest
C6 | shortest
C6 | fastest
C6 | safest
C7 | shortest
C7 | fastest
C7 | safest
C8 | shortest
C8 | fastest
C8 | safest
C9 | shortest
C9 | fastest
C9 | safest
C10 | shortest
C10 | fastest
C10 | safest
L47 -> DEPOT | shortest
L172 -> DEPOT | shortest
L101 -> DEPOT | shortest
L47 -> DEPOT | fastest
L172 -> DEPOT | fastest
L101 -> DEPOT | fastest
L47 -> DEPOT | safest
L172 -> DEPOT | safest
L101 -> DEPOT | safest
L72 -> C1 | shortest
L153 -> C1 | shortest
L12 -> C1 | shortest
L72 -> C1 | fastest
L153 -> C1 | fastest
L12 -> C1 | fastest
L72 -> C1 | safest
L153 -> C

Medium

In [ ]:

import pickle
import gzip
import pandas as pd
import numpy as np
import networkx as nx

# CONFIG

OUTPUT_CSV = "od_matrix_medium_charger.csv"

BOUNDING_BUFFER_DEG = 1.0
MAX_CHARGERS_PER_CUSTOMER = 3


# POINTS 

POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.367289, "lon": 12.434819, "type": "customer"},
    {"id": "C2",    "lat": 51.483631, "lon": 12.015296, "type": "customer"},
    {"id": "C3",    "lat": 51.084678, "lon": 13.768165, "type": "customer"},
    {"id": "C4",    "lat": 50.798920, "lon": 12.929745, "type": "customer"},
    {"id": "C5",    "lat": 50.979200, "lon": 11.125600, "type": "customer"},
    {"id": "C6",    "lat": 50.945592, "lon": 11.608864, "type": "customer"},
    {"id": "C7",    "lat": 50.961200, "lon": 11.259260, "type": "customer"},
    {"id": "C8",    "lat": 50.912492, "lon": 12.061403, "type": "customer"},
    {"id": "C9",    "lat": 52.101822, "lon": 11.630092, "type": "customer"},
    {"id": "C10",   "lat": 51.802704, "lon": 12.235987, "type": "customer"},
    {"id": "C11",   "lat": 51.629473, "lon": 12.301097, "type": "customer"},
    {"id": "C12",   "lat": 51.769678, "lon": 14.364260, "type": "customer"},
    {"id": "C13",   "lat": 50.709001, "lon": 12.481034, "type": "customer"},
    {"id": "C14",   "lat": 50.499501, "lon": 12.147448, "type": "customer"},
    {"id": "C15",   "lat": 50.304638, "lon": 11.919935, "type": "customer"},
    {"id": "C16",   "lat": 51.486191, "lon": 10.791956, "type": "customer"},
    {"id": "C17",   "lat": 51.884395, "lon": 11.078309, "type": "customer"},
    {"id": "C18",   "lat": 51.876171, "lon": 12.591305, "type": "customer"},
    {"id": "C19",   "lat": 51.552248, "lon": 12.970863, "type": "customer"},
    {"id": "C20",   "lat": 51.146041, "lon": 11.832647, "type": "customer"},
]

points_df = pd.DataFrame(POINTS)



# PROFILE FUNKTION

def weight_func(profile):

    def w(u, v, d):

        if profile == "safest" and d.get("tunnel", "no") != "no":
            return float("inf")

        if profile == "shortest":
            return d["dist_km"]

        if profile == "fastest":
            return d["time_h"]

        if profile == "safest":
            return d["dist_km"] + 2.0 * d["risk"] + 0.5 * d["road_penalty"]

        return d["dist_km"]

    return w

# GRAPH LADEN

edges = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_medium.parquet"
)

charging_hubs = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\charging_hubs.medium.parquet"
)

with gzip.open(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_medium.pkl.gz",
    "rb"
) as f:
    node_coords = pickle.load(f)

# GRAPH BUILD

G = nx.DiGraph()

for edge in edges.itertuples(index=False):

    u = int(edge.u)
    v = int(edge.v)

    dist_km = float(edge.dist) / 1000.0

    speed = 70
    time_h = dist_km / speed

    risk = (
        float(getattr(edge, "accident_score_norm", 0.0))
        * float(getattr(edge, "pop_score_norm", 0.0))
    )

    highway = int(getattr(edge, "highway", 12))

    if highway <= 6:
        road_penalty = 0.0
    else:
        road_penalty = 0.3

    tunnel = getattr(edge, "tunnel", "no")

    G.add_edge(
        u,
        v,
        dist_km=dist_km,
        time_h=time_h,
        risk=risk,
        road_penalty=road_penalty,
        tunnel=tunnel
    )


print(f"Graph: {G.number_of_nodes()} nodes")

# BOUNDING BOX

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# LADESAULEN REDUZIEREN

print("\nLadesäulen auswählen...")

charger_points = []

charging_hubs = charging_hubs.to_dict("records")

for i, ch in enumerate(charging_hubs):

    lat, lon = node_coords[ch["node"]]

    if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
        charger_points.append({
            "id": f"L{i}",
            "lat": lat,
            "lon": lon,
            "node": ch["node"],
            "type": "charger"
        })

print(f"Ladesäulen in Box: {len(charger_points)}")

selected_ids = set()

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (loc["lat"] - ch["lat"])**2 + (loc["lon"] - ch["lon"])**2
        dists.append((d, ch["id"]))

    dists.sort()

    nearest = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

    selected_ids.update(nearest)

charger_points = [ch for ch in charger_points if ch["id"] in selected_ids]

customer_to_chargers = {}

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (
            (loc["lat"] - ch["lat"])**2 +
            (loc["lon"] - ch["lon"])**2
        )

        dists.append((d, ch["id"]))

    dists.sort()

    customer_to_chargers[loc["id"]] = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

print(f"Reduzierte Ladesäulen: {len(charger_points)}")

points_df = pd.concat([points_df, pd.DataFrame(charger_points)], ignore_index=True)



# NEAREST NODE

node_ids = np.array(list(node_coords.keys()))
node_lats = np.array([node_coords[n][0] for n in node_ids])
node_lons = np.array([node_coords[n][1] for n in node_ids])

def nearest(lat, lon):
    d = (node_lats - lat)**2 + (node_lons - lon)**2
    return int(node_ids[np.argmin(d)])

if "node" not in points_df.columns:
    points_df["node"] = np.nan

points_df["node"] = points_df["node"].fillna(
    points_df.apply(
        lambda r: nearest(r.lat, r.lon),
        axis=1
    )
)

customers_df = points_df[
    points_df["type"].isin(["customer", "depot"])
].copy()

chargers_df = points_df[
    points_df["type"] == "charger"
].copy()

charger_lookup = {
    row["id"]: row
    for _, row in chargers_df.iterrows()
}

print(points_df["type"].value_counts())
print("\nBerechne OD Matrix...")

records = []
routes = {}


# CUSTOMER -> CHARGER

for _, src in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        print(f"{src.id} | {profile}")

        for charger_id in customer_to_chargers[src.id]:

            dst = charger_lookup[charger_id]

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type,
                "to_type": dst.type,
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })


# CHARGER -> CUSTOMER

for _, dst in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        allowed_chargers = customer_to_chargers[dst.id]

        for charger_id in allowed_chargers:

            src = charger_lookup[charger_id]

            print(f"{src.id} -> {dst.id} | {profile}")

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type if hasattr(src, "type") else "customer",
                "to_type": dst.type if hasattr(dst, "type") else "customer",
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
with open("routes_medium_charger.pkl", "wb") as f:
    pickle.dump(
        routes,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("\n✅ Fertig!")
print("Matrixgröße:", len(df))


Graph: 608056 nodes

Ladesäulen auswählen...
Ladesäulen in Box: 627
Reduzierte Ladesäulen: 63
type
charger     63
customer    20
depot        1
Name: count, dtype: int64

Berechne OD Matrix...
DEPOT | shortest
DEPOT | fastest
DEPOT | safest
C1 | shortest
C1 | fastest
C1 | safest
C2 | shortest
C2 | fastest
C2 | safest
C3 | shortest
C3 | fastest
C3 | safest
C4 | shortest
C4 | fastest
C4 | safest
C5 | shortest
C5 | fastest
C5 | safest
C6 | shortest
C6 | fastest
C6 | safest
C7 | shortest
C7 | fastest
C7 | safest
C8 | shortest
C8 | fastest
C8 | safest
C9 | shortest
C9 | fastest
C9 | safest
C10 | shortest
C10 | fastest
C10 | safest
C11 | shortest
C11 | fastest
C11 | safest
C12 | shortest
C12 | fastest
C12 | safest
C13 | shortest
C13 | fastest
C13 | safest
C14 | shortest
C14 | fastest
C14 | safest
C15 | shortest
C15 | fastest
C15 | safest
C16 | shortest
C16 | fastest
C16 | safest
C17 | shortest
C17 | fastest
C17 | safest
C18 | shortest
C18 | fastest
C18 | safest
C19 | shortest
C19 | fastest
C

Large

In [ ]:
import pickle
import gzip
import pandas as pd
import numpy as np
import networkx as nx

# CONFIG

OUTPUT_CSV = "od_matrix_large_charger.csv"

BOUNDING_BUFFER_DEG = 1.0
MAX_CHARGERS_PER_CUSTOMER = 3


# POINTS 

POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.367289, "lon": 12.434819, "type": "customer"},
    {"id": "C2",    "lat": 51.483631, "lon": 12.015296, "type": "customer"},
    {"id": "C3",    "lat": 51.084678, "lon": 13.768165, "type": "customer"},
    {"id": "C4",    "lat": 50.798920, "lon": 12.929745, "type": "customer"},
    {"id": "C5",    "lat": 50.979200, "lon": 11.125600, "type": "customer"},
    {"id": "C6",    "lat": 50.945592, "lon": 11.608864, "type": "customer"},
    {"id": "C7",    "lat": 50.961200, "lon": 11.259260, "type": "customer"},
    {"id": "C8",    "lat": 50.912492, "lon": 12.061403, "type": "customer"},
    {"id": "C9",    "lat": 52.101822, "lon": 11.630092, "type": "customer"},
    {"id": "C10",   "lat": 51.802704, "lon": 12.235987, "type": "customer"},
    {"id": "C11",   "lat": 51.629473, "lon": 12.301097, "type": "customer"},
    {"id": "C12",   "lat": 51.769678, "lon": 14.364260, "type": "customer"},
    {"id": "C13",   "lat": 50.709001, "lon": 12.481034, "type": "customer"},
    {"id": "C14",   "lat": 50.499501, "lon": 12.147448, "type": "customer"},
    {"id": "C15",   "lat": 50.304638, "lon": 11.919935, "type": "customer"},
    {"id": "C16",   "lat": 51.486191, "lon": 10.791956, "type": "customer"},
    {"id": "C17",   "lat": 51.884395, "lon": 11.078309, "type": "customer"},
    {"id": "C18",   "lat": 51.876171, "lon": 12.591305, "type": "customer"},
    {"id": "C19",   "lat": 51.552248, "lon": 12.970863, "type": "customer"},
    {"id": "C20",   "lat": 51.146041, "lon": 11.832647, "type": "customer"},
    {"id": "C21",   "lat": 51.342678, "lon": 11.988740, "type": "customer"},
    {"id": "C22",   "lat": 51.184292, "lon": 12.017954, "type": "customer"},
    {"id": "C23",   "lat": 51.411562, "lon": 12.172039, "type": "customer"},
    {"id": "C24",   "lat": 51.512967, "lon": 12.337281, "type": "customer"},
    {"id": "C25",   "lat": 51.358887, "lon": 12.766669, "type": "customer"},
    {"id": "C26",   "lat": 51.220925, "lon": 12.718451, "type": "customer"},
    {"id": "C27",   "lat": 51.129939, "lon": 12.512646, "type": "customer"},
    {"id": "C28",   "lat": 51.001292, "lon": 12.455299, "type": "customer"},
    {"id": "C29",   "lat": 51.159790, "lon": 13.517739, "type": "customer"},
    {"id": "C30",   "lat": 51.314591, "lon": 13.280383, "type": "customer"},
    {"id": "C31",   "lat": 50.921715, "lon": 13.392756, "type": "customer"},
    {"id": "C32",   "lat": 51.168095, "lon": 14.413691, "type": "customer"},
    {"id": "C33",   "lat": 51.430276, "lon": 14.289119, "type": "customer"},
    {"id": "C34",   "lat": 51.527821, "lon": 14.016959, "type": "customer"},
    {"id": "C35",   "lat": 51.513295, "lon": 11.579354, "type": "customer"},
    {"id": "C36",   "lat": 51.480816, "lon": 11.315494, "type": "customer"},
    {"id": "C37",   "lat": 51.769277, "lon": 11.483116, "type": "customer"},
    {"id": "C38",   "lat": 51.798561, "lon": 11.755918, "type": "customer"},
    {"id": "C39",   "lat": 51.746434, "lon": 12.002407, "type": "customer"},
    {"id": "C40",   "lat": 51.867664, "lon": 11.557193, "type": "customer"},
    {"id": "C41",   "lat": 51.850656, "lon": 10.787768, "type": "customer"},
    {"id": "C42",   "lat": 51.790323, "lon": 11.168981, "type": "customer"},
    {"id": "C43",   "lat": 51.385616, "lon": 10.848824, "type": "customer"},
    {"id": "C44",   "lat": 51.198762, "lon": 10.478537, "type": "customer"},
    {"id": "C45",   "lat": 50.927969, "lon": 10.715398, "type": "customer"},
    {"id": "C46",   "lat": 50.867329, "lon": 10.948139, "type": "customer"},
    {"id": "C47",   "lat": 50.697818, "lon": 10.937663, "type": "customer"},
    {"id": "C48",   "lat": 50.560132, "lon": 10.371962, "type": "customer"},
    {"id": "C49",   "lat": 50.243487, "lon": 10.967718, "type": "customer"},
    {"id": "C50",   "lat": 49.910244, "lon": 10.869676, "type": "customer"},
]

points_df = pd.DataFrame(POINTS)



# PROFILE FUNKTION

def weight_func(profile):

    def w(u, v, d):

        if profile == "safest" and d.get("tunnel", "no") != "no":
            return float("inf")

        if profile == "shortest":
            return d["dist_km"]

        if profile == "fastest":
            return d["time_h"]

        if profile == "safest":
            return d["dist_km"] + 2.0 * d["risk"] + 0.5 * d["road_penalty"]

        return d["dist_km"]

    return w

# GRAPH LADEN

edges = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\graph_final_core_large.parquet"
)

charging_hubs = pd.read_parquet(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\charging_hubs.large.parquet"
)

with gzip.open(
    r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\node_coords_large.pkl.gz",
    "rb"
) as f:
    node_coords = pickle.load(f)

# GRAPH BUILD

G = nx.DiGraph()

for edge in edges.itertuples(index=False):

    u = int(edge.u)
    v = int(edge.v)

    dist_km = float(edge.dist) / 1000.0

    speed = 70
    time_h = dist_km / speed

    risk = (
        float(getattr(edge, "accident_score_norm", 0.0))
        * float(getattr(edge, "pop_score_norm", 0.0))
    )

    highway = int(getattr(edge, "highway", 12))

    if highway <= 6:
        road_penalty = 0.0
    else:
        road_penalty = 0.3

    tunnel = getattr(edge, "tunnel", "no")

    G.add_edge(
        u,
        v,
        dist_km=dist_km,
        time_h=time_h,
        risk=risk,
        road_penalty=road_penalty,
        tunnel=tunnel
    )


print(f"Graph: {G.number_of_nodes()} nodes")

# BOUNDING BOX

lat_min = points_df.lat.min() - BOUNDING_BUFFER_DEG
lat_max = points_df.lat.max() + BOUNDING_BUFFER_DEG
lon_min = points_df.lon.min() - BOUNDING_BUFFER_DEG
lon_max = points_df.lon.max() + BOUNDING_BUFFER_DEG

# LADESAULEN REDUZIEREN

print("\nLadesäulen auswählen...")

charger_points = []

charging_hubs = charging_hubs.to_dict("records")

for i, ch in enumerate(charging_hubs):

    lat, lon = node_coords[ch["node"]]

    if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
        charger_points.append({
            "id": f"L{i}",
            "lat": lat,
            "lon": lon,
            "node": ch["node"],
            "type": "charger"
        })

print(f"Ladesäulen in Box: {len(charger_points)}")

selected_ids = set()

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (loc["lat"] - ch["lat"])**2 + (loc["lon"] - ch["lon"])**2
        dists.append((d, ch["id"]))

    dists.sort()

    nearest = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

    selected_ids.update(nearest)

charger_points = [ch for ch in charger_points if ch["id"] in selected_ids]

customer_to_chargers = {}

for _, loc in points_df[
    points_df["type"].isin(["customer", "depot"])
].iterrows():

    dists = []

    for ch in charger_points:
        d = (
            (loc["lat"] - ch["lat"])**2 +
            (loc["lon"] - ch["lon"])**2
        )

        dists.append((d, ch["id"]))

    dists.sort()

    customer_to_chargers[loc["id"]] = [
        cid
        for _, cid in dists[:MAX_CHARGERS_PER_CUSTOMER]
    ]

print(f"Reduzierte Ladesäulen: {len(charger_points)}")

points_df = pd.concat([points_df, pd.DataFrame(charger_points)], ignore_index=True)



# NEAREST NODE

node_ids = np.array(list(node_coords.keys()))
node_lats = np.array([node_coords[n][0] for n in node_ids])
node_lons = np.array([node_coords[n][1] for n in node_ids])

def nearest(lat, lon):
    d = (node_lats - lat)**2 + (node_lons - lon)**2
    return int(node_ids[np.argmin(d)])

if "node" not in points_df.columns:
    points_df["node"] = np.nan

points_df["node"] = points_df["node"].fillna(
    points_df.apply(
        lambda r: nearest(r.lat, r.lon),
        axis=1
    )
)

customers_df = points_df[
    points_df["type"].isin(["customer", "depot"])
].copy()

chargers_df = points_df[
    points_df["type"] == "charger"
].copy()

charger_lookup = {
    row["id"]: row
    for _, row in chargers_df.iterrows()
}

print(points_df["type"].value_counts())
print("\nBerechne OD Matrix...")

records = []
routes = {}


# CUSTOMER -> CHARGER

for _, src in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        print(f"{src.id} | {profile}")

        for charger_id in customer_to_chargers[src.id]:

            dst = charger_lookup[charger_id]

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type,
                "to_type": dst.type,
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })


# CHARGER -> CUSTOMER

for _, dst in customers_df.iterrows():

    for profile in ["shortest", "fastest", "safest"]:

        allowed_chargers = customer_to_chargers[dst.id]

        for charger_id in allowed_chargers:

            src = charger_lookup[charger_id]

            print(f"{src.id} -> {dst.id} | {profile}")

            try:
                length, path = nx.single_source_dijkstra(
                    G,
                    src.node,
                    dst.node,
                    weight=weight_func(profile)
                )

            except nx.NetworkXNoPath:
                continue

            routes[(src.id, dst.id, profile)] = path

            dist_val = 0
            time_val = 0
            risk_val = 0
            tunnel_used = False

            for i in range(len(path) - 1):

                e = G[path[i]][path[i + 1]]

                if e.get("tunnel", "no") != "no":
                    tunnel_used = True

                dist_val += e["dist_km"]
                time_val += e["time_h"]
                risk_val += e["risk"]

            records.append({
                "from": src.id,
                "to": dst.id,
                "profile": profile,
                "tunnel_used": tunnel_used,
                "from_type": src.type if hasattr(src, "type") else "customer",
                "to_type": dst.type if hasattr(dst, "type") else "customer",
                "dist_km": dist_val,
                "time_min": time_val * 60,
                "risk": risk_val
            })

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
with open("routes_large_charger.pkl", "wb") as f:
    pickle.dump(
        routes,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("\n✅ Fertig!")
print("Matrixgröße:", len(df))


Graph: 953326 nodes

Ladesäulen auswählen...
Ladesäulen in Box: 814
Reduzierte Ladesäulen: 146
type
charger     146
customer     50
depot         1
Name: count, dtype: int64

Berechne OD Matrix...
DEPOT | shortest
DEPOT | fastest
DEPOT | safest
C1 | shortest
C1 | fastest
C1 | safest
C2 | shortest
C2 | fastest
C2 | safest
C3 | shortest
C3 | fastest
C3 | safest
C4 | shortest
C4 | fastest
C4 | safest
C5 | shortest
C5 | fastest
C5 | safest
C6 | shortest
C6 | fastest
C6 | safest
C7 | shortest
C7 | fastest
C7 | safest
C8 | shortest
C8 | fastest
C8 | safest
C9 | shortest
C9 | fastest
C9 | safest
C10 | shortest
C10 | fastest
C10 | safest
C11 | shortest
C11 | fastest
C11 | safest
C12 | shortest
C12 | fastest
C12 | safest
C13 | shortest
C13 | fastest
C13 | safest
C14 | shortest
C14 | fastest
C14 | safest
C15 | shortest
C15 | fastest
C15 | safest
C16 | shortest
C16 | fastest
C16 | safest
C17 | shortest
C17 | fastest
C17 | safest
C18 | shortest
C18 | fastest
C18 | safest
C19 | shortest
C19 | faste